# 🦾 Master Arm Teleoperation — Dataset Collection Pipeline

## 개요
이 노트북은 **Master Arm**을 사용하여 RBY1 로봇을 원격 조작(Teleoperation)하고,  
데모 데이터를 **H5 파일**로 수집하는 파이프라인을 구현합니다.

## 🌐 시스템 아키텍처

```
[Robot PC (UPC) — 192.168.0.56]          [Remote PC — 이 노트북]
  master_arm_server.py                     data_collection_master_arm.ipynb
  ├─ rby.upc.MasterArm (로컬 USB)   UDP    ├─ RemoteMasterArm 클라이언트
  │  └─ 중력 보상 / homing 루프  ◄──────► │  └─ 관절값/버튼 수신 (port 5010/5011)
  │                                        │
  gripper_server.py                        ├─ RemoteGripper 클라이언트
  ├─ rby.DynamixelBus (로컬 USB)    UDP    │  └─ 그리퍼 제어 (port 5009)
  │  └─ Dynamixel 그리퍼 제어    ◄──────► │
  │                                        ├─ rby.create_robot() → 로봇 명령
  └─ (gRPC robot: 192.168.0.50:50051)      ├─ MultiRealsense → 카메라
                                           └─ H5Writer → 데이터 저장
```

| 구성 요소 | 실행 위치 | 통신 방식 |
|---|---|---|
| `master_arm_server.py` | **Robot PC** | 실행 (`python master_arm_server.py`) |
| `gripper_server.py` | **Robot PC** | 실행 (`python gripper_server.py`) |
| `rby.create_robot()` | Remote PC | gRPC 네트워크 (`192.168.0.50:50051`) |
| `RemoteMasterArm` | Remote PC | UDP (`state_port=5010`, `cmd_port=5011`) |
| `RemoteGripper` | Remote PC | UDP (`port=5009`) |
| `MultiRealsense` | Remote PC | USB 직접 |

### ▶ 사전 준비: Robot PC에서 두 서버 모두 실행

```bash
# Robot PC (UPC) 터미널 1:
cd /path/to/rby1-data-collection
python master_arm_server.py \
    --device /dev/rby1_master_arm \
    --urdf ../rby1-sdk/models/master_arm/model.urdf \
    --state-port 5010 \
    --cmd-port 5011

# Robot PC (UPC) 터미널 2:
cd /path/to/rby1-data-collection
python gripper_server.py --port 5009
```

### 실행 순서
| 단계 | 셀 | 내용 |
|---|---|---|
| 0 | (사전) | Robot PC에서 `master_arm_server.py` + `gripper_server.py` 실행 |
| 1 | Step 1 | 로봇 연결 |
| 2 | Step 2 | RemoteMasterArm 연결 (서버 ping 확인) |
| 3 | Step 3 | 그리퍼 초기화 (선택) |
| 4 | Step 4 | 카메라 초기화 |
| 5 | **Step 5** | 로봇 초기 자세 이동 |
| 6 | **Step 5.5** | ⭐ Master Arm 자동 초기 자세 이동 (서버에 homing 명령) |
| 7 | **Step 6** | 텔레오퍼레이션 시작 |
| 8 | **Step 7** | 데이터 수집 UI |

### Master Arm 버튼 매핑

| 물리 버튼 | SDK 필드 | 동작 |
|---|---|---|
| 오른쪽 **트리거** | `button_right.trigger` | 오른쪽 그리퍼 열기↔닫기 토글 |
| 왼쪽 **트리거** | `button_left.trigger` | 왼쪽 그리퍼 열기↔닫기 토글 |
| 오른쪽 **잠금 해제 버튼** | `button_right.button` | (그리퍼 제어 안 함) |
| 왼쪽 **잠금 해제 버튼** | `button_left.button` | (그리퍼 제어 안 함) |


In [ ]:
# [Step 0] 라이브러리 임포트 및 sys.path 설정
import os
import sys
import time
import threading
import logging
import queue
import yaml
import numpy as np
import cv2
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import ipywidgets as widgets
from IPython.display import display, HTML

# ──────────────────────────────────────────────────────────
# sys.path: rby1-data-collection 모듈들을 import하기 위해 추가
# ──────────────────────────────────────────────────────────
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_COLLECTION_DIR = os.path.abspath(os.path.join(_NOTEBOOK_DIR, ".."))
if DATA_COLLECTION_DIR not in sys.path:
    sys.path.insert(0, DATA_COLLECTION_DIR)

# ──────────────────────────────────────────────────────────
# RBY1 SDK + 데이터 수집 모듈
# ──────────────────────────────────────────────────────────
import rby1_sdk as rby
from h5_writer import H5Writer
from camera import MultiRealsense
from remote_gripper import Gripper as RemoteGripper
from remote_master_arm import RemoteMasterArm          # ← UDP 클라이언트 (Robot PC 서버와 통신)
from utils import get_next_h5_path
from pcd_utils import rgbd_to_pointcloud, REALSENSE_D435_INTRINSICS, REALSENSE_D435_INTRINSICS_848x480

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)-8s - %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

print("✅ 패키지 임포트 완료")
print(f"   DATA_COLLECTION_DIR : {DATA_COLLECTION_DIR}")
print(f"   rby1_sdk 버전       : {rby.__version__ if hasattr(rby, '__version__') else 'unknown'}")


In [ ]:
# [Step 0] config.yaml 로드 및 전역 설정값 정의
# ══════════════════════════════════════════════════════════
# ⚙️  설정값 (필요에 따라 수정)
# ══════════════════════════════════════════════════════════

CONFIG_PATH = os.path.join(DATA_COLLECTION_DIR, "config.yaml")

with open(CONFIG_PATH, encoding="utf-8") as _f:
    CONFIG = yaml.safe_load(_f)

# ── 로봇 연결 (네트워크 gRPC) ──────────────────────────────
ROBOT_ADDRESS = CONFIG.get("user_pc_ip", "192.168.0.50:50051")
ROBOT_MODEL   = "a"   # 'a' = RBY1 Model_A

# ── Master Arm 서버 (Robot PC에서 실행 중인 master_arm_server.py) ──
ROBOT_PC_IP           = CONFIG.get("remote_master_arm_host",
                            CONFIG.get("remote_gripper_host", "192.168.0.56"))
MA_STATE_PORT         = int(CONFIG.get("remote_master_arm_state_port", 5010))
MA_CMD_PORT           = int(CONFIG.get("remote_master_arm_cmd_port",   5011))

# ── 녹화 ─────────────────────────────────────────────────
REC_FPS    = int(CONFIG.get("rec_fps", 15))
DEMO_ROOT  = os.path.join(
    CONFIG.get("demo_root", "/media/hyunjin/T7/rby1_demo/"),
    CONFIG.get("task_name", "task"),
)
os.makedirs(DEMO_ROOT, exist_ok=True)

# ── 카메라 ────────────────────────────────────────────────
IMG_WIDTH  = CONFIG.get("img_size", {}).get("width",  640)
IMG_HEIGHT = CONFIG.get("img_size", {}).get("height", 480)
CAMERAS_CFG = CONFIG.get("cameras", {}) or {}
CAM_NAMES   = list(CAMERAS_CFG.keys())
CAM_SERIALS = {name: spec["serial"] for name, spec in CAMERAS_CFG.items()}
PRIMARY_SERIAL = CAM_SERIALS[CAM_NAMES[0]] if CAM_NAMES else None
CAMERA_WARMUP_SECS   = float(CONFIG.get("camera_warmup_secs", 2.0))
CAMERA_WARMUP_FRAMES = int(CONFIG.get("camera_warmup_frames", max(15, REC_FPS)))

# ── Master Arm 관절 매핑 ──────────────────────────────────
# RBY1 Bimanual Master Arm: q_joint[0:7] = 오른팔, q_joint[7:14] = 왼팔
MASTER_RIGHT_SLICE = slice(0, 7)
MASTER_LEFT_SLICE  = slice(7, 14)

# ── 제어 주기 ─────────────────────────────────────────────
CONTROL_DT = 0.01   # 100 Hz

# ── 로봇 초기 자세 (단위: degree) ──────────────────────────
TORSO_INIT_DEG     = [0.0,  20.0, -40.0,  35.0,   0.0,  0.0]
RIGHT_ARM_INIT_DEG = [-24.0, -60.0,  10.0, -120.0, -60.0, 85.0, 0.0]
LEFT_ARM_INIT_DEG  = [-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0]
HEAD_INIT_DEG      = [0.0, 40.0]

# ── 그리퍼 ───────────────────────────────────────────────
USE_GRIPPER = True   # 그리퍼 없이 실행하려면 False로 변경

print("✅ 설정 로드 완료")
print(f"   ROBOT_ADDRESS      : {ROBOT_ADDRESS}")
print(f"   Master Arm 서버    : {ROBOT_PC_IP}  (state:{MA_STATE_PORT} / cmd:{MA_CMD_PORT})")
print(f"   DEMO_ROOT          : {DEMO_ROOT}")
print(f"   REC_FPS            : {REC_FPS} Hz")
print(f"   카메라 목록        : {CAM_NAMES}")
print()
print("📌 Robot PC에서 master_arm_server.py가 실행 중인지 확인하세요:")
print(f"   python master_arm_server.py --state-port {MA_STATE_PORT} --cmd-port {MA_CMD_PORT}")


In [ ]:
# [Step 0] SharedState 전역 클래스 및 하드웨어 핸들 정의
# ══════════════════════════════════════════════════════════
# 📦 전역 상태 클래스 정의 (thread-safe)
# ══════════════════════════════════════════════════════════

@dataclass
class SharedState:
    """Master Arm 텔레오퍼레이션에서 모든 스레드가 공유하는 상태."""

    lock: threading.Lock = field(default_factory=threading.Lock)

    # ── 로봇 상태 (robot state callback이 갱신) ──────────────
    robot_joint_positions: np.ndarray = field(
        default_factory=lambda: np.array([])
    )

    # ── Master Arm 상태 ───────────────────────────────────
    master_arm_q:       Optional[np.ndarray] = None   # 관절 각도 (14,)
    master_arm_gravity: Optional[np.ndarray] = None   # 중력 보상 토크 (14,)

    # ── 그리퍼 [right, left], 규칙: 0=CLOSED / 1=OPEN ─────
    #
    # gripper_target: Master Arm 트리거 r_norm
    #   r_norm=0 (트리거 미입력) → CLOSED,  r_norm=1 (완전 입력) → OPEN
    #
    # gripper_state:  실제 엔코더 위치 (1 - get_state() 반전 적용)
    #   get_state(): 0=OPEN, 1=CLOSED → 1-x 반전 → 0=CLOSED, 1=OPEN
    #
    # → 두 값 모두 동일 규칙: 0=CLOSED, 1=OPEN
    #
    # ※ gripper_state는 state_poll_thread_fn이 주기적으로 갱신.
    #   데이터 로거는 이 캐시를 읽으므로 별도 UDP 호출 불필요.
    gripper_target: np.ndarray = field(default_factory=lambda: np.array([0.0, 0.0]))
    gripper_state:  np.ndarray = field(default_factory=lambda: np.array([0.0, 0.0]))

    # ── 잠금 해제 버튼 (Deadman switch) ──────────────────────
    # .button 필드: 누른 동안만 팔로우, 놓으면 위치 고정
    unlock_right_active: bool = False
    unlock_left_active:  bool = False
    unlock_right_prev:   bool = False   # 에지 감지용
    unlock_left_prev:    bool = False

    # ── Hold 위치 (unlock 버튼 놓는 순간 로봇 실제 위치 저장) ─
    hold_right_q: Optional[np.ndarray] = None
    hold_left_q:  Optional[np.ndarray] = None

    # ── 제어 플래그 ────────────────────────────────────────
    is_teleop_active: bool = False

    # ── 카메라 워밍업 ─────────────────────────────────────
    camera_warmup_done: bool = False

    # ── 녹화 상태 ─────────────────────────────────────────
    is_recording:         bool = False
    h5_writer:            Optional[object] = None
    rec_stop_event:       Optional[threading.Event] = None
    current_episode_path: Optional[str] = None


# 전역 싱글턴
shared_state = SharedState()

# 전역 하드웨어 핸들
robot       : Optional[rby.Robot_A]       = None
robot_model : Optional[rby.Model_A]       = None
master_arm  : Optional[RemoteMasterArm]   = None
gripper     : Optional[RemoteGripper]     = None
realsense   : Optional[MultiRealsense]    = None

print("✅ SharedState 및 전역 변수 정의 완료")


---
## Step 1 : 로봇 연결

RBY1 로봇에 연결하고 서보/컨트롤 매니저를 활성화합니다.  
연결이 완료되면 `robot_state_callback`이 주기적으로 관절 위치를 `shared_state`에 업데이트합니다.


In [ ]:
# [Step 1] 로봇 연결 및 Control Manager 활성화
def robot_state_callback(robot_state: rby.RobotState_A):
    """로봇 상태 콜백 — joint_positions를 shared_state에 저장."""
    with shared_state.lock:
        shared_state.robot_joint_positions = np.array(robot_state.position, dtype=float)


def connect_robot(address: str, model: str = "a") -> rby.Robot_A:
    global _robot, robot_model

    _robot = rby.create_robot(address, model)
    if not _robot.connect():
        raise RuntimeError(f"로봇 연결 실패: {address}")

    # 파워 온
    if not _robot.is_power_on(".*"):
        if not _robot.power_on(".*"):
            raise RuntimeError("로봇 파워 온 실패")

    # 서보 온
    if not _robot.is_servo_on(".*"):
        if not _robot.servo_on(".*"):
            raise RuntimeError("서보 온 실패")

    # Control Manager Fault 처리
    cm_state = _robot.get_control_manager_state().state
    if cm_state in (
        rby.ControlManagerState.State.MajorFault,
        rby.ControlManagerState.State.MinorFault,
    ):
        logger.warning(f"Control Manager Fault ({cm_state.name}) → 리셋 시도")
        if not _robot.reset_fault_control_manager():
            raise RuntimeError("Control Manager 리셋 실패")

    if not _robot.enable_control_manager(unlimited_mode_enabled=True):
        raise RuntimeError("Control Manager 활성화 실패")

    robot_model = _robot.model()

    # SDK 패턴: tool flange 12V 공급 (그리퍼 전원)
    for arm in ["right", "left"]:
        if not _robot.set_tool_flange_output_voltage(arm, 12):
            logger.warning(f"[{arm}] tool flange 12V 공급 실패")

    # SDK 패턴: 관절 위치 명령 저주파 필터 설정 (부드러운 추종)
    _robot.set_parameter("joint_position_command.cutoff_frequency", "3")

    # 관절 상태 주기적 수신 (100 Hz)
    _robot.start_state_update(robot_state_callback, 1.0 / CONTROL_DT)

    logger.info(f"✅ 로봇 연결 완료: {address}")
    return _robot


# ── 실행 ──────────────────────────────────────────────────
robot = connect_robot(ROBOT_ADDRESS, ROBOT_MODEL)
print(f"✅ 로봇 연결 완료")
print(f"   헤드 DOF  : {len(robot_model.head_idx)}")
print(f"   몸통 DOF  : {len(robot_model.torso_idx)}")
print(f"   오른팔 DOF: {len(robot_model.right_arm_idx)}")
print(f"   왼팔 DOF  : {len(robot_model.left_arm_idx)}")


---
## Step 2 : RemoteMasterArm 연결

`master_arm_server.py`가 Robot PC에서 실행 중이어야 합니다.

### Robot PC에서 서버 실행 (아직 안 했다면)
```bash
cd /path/to/rby1-data-collection
python master_arm_server.py \
    --device /dev/rby1_master_arm \
    --urdf ../rby1-sdk/models/master_arm/model.urdf \
    --state-port 5010 \
    --cmd-port 5011
```

이 셀은 서버에 ping을 보내 연결을 확인하고, 상태 수신 스레드를 시작합니다.


In [ ]:
# [Step 2-1] master_arm_server.py 연결 진단 (네트워크 + UDP ping)
# ══════════════════════════════════════════════════════════
# 🔍 master_arm_server.py 연결 진단
# ══════════════════════════════════════════════════════════
import socket, subprocess

def diagnose_master_arm_server(host: str, cmd_port: int, state_port: int, timeout: float = 3.0):
    print("=" * 60)
    print("  Master Arm Server 연결 진단")
    print("=" * 60)
    print(f"\n  서버 주소  : {host}")
    print(f"  명령 포트  : {cmd_port}")
    print(f"  상태 포트  : {state_port}")

    # 1) 네트워크 ping
    try:
        r = subprocess.run(["ping", "-c", "1", "-W", "1", host],
                           capture_output=True, text=True, timeout=5)
        reachable = r.returncode == 0
        print(f"\n[1] 네트워크 ping    : {'✅ 응답' if reachable else '❌ 응답 없음'}")
    except Exception as e:
        reachable = False
        print(f"\n[1] 네트워크 ping    : ❌ 실패 ({e})")

    # 2) UDP ping (cmd_port)
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)
    server_ok = False
    try:
        import json, time
        payload = json.dumps({"cmd": "ping", "ts": time.time()}).encode()
        sock.sendto(payload, (host, cmd_port))
        raw, _ = sock.recvfrom(65535)
        resp   = json.loads(raw.decode())
        server_ok = resp.get("ok", False)
        print(f"[2] master_arm_server: {'✅ 응답 OK' if server_ok else f'❌ 응답 이상: {resp}'}")
    except socket.timeout:
        print(f"[2] master_arm_server: ❌ 타임아웃 — 서버가 실행 중인지 확인하세요")
    except Exception as e:
        print(f"[2] master_arm_server: ❌ 오류 ({e})")
    finally:
        sock.close()

    print("\n" + "=" * 60)
    if server_ok:
        print("✅ 서버 연결 확인 완료 — Step 2 init 셀을 실행하세요.")
    else:
        print("⚠️  서버 연결 실패. 아래를 확인하세요:")
        print(f"   1. Robot PC({host})에서 master_arm_server.py 실행 중인지 확인")
        print(f"   2. 방화벽: sudo ufw allow {cmd_port}/udp && sudo ufw allow {state_port}/udp")
        print(f"   3. config.yaml의 remote_master_arm_host 값이 올바른지 확인")

    return server_ok


_server_ok = diagnose_master_arm_server(ROBOT_PC_IP, MA_CMD_PORT, MA_STATE_PORT)


In [ ]:
# [Step 2-2] RemoteMasterArm 클라이언트 초기화 및 연결
def init_master_arm() -> RemoteMasterArm:
    """
    RemoteMasterArm 클라이언트를 생성하고 master_arm_server.py(Robot PC)에 연결합니다.
    rby.upc.MasterArm은 Robot PC의 master_arm_server.py 내부에서 처리됩니다.
    """
    global master_arm

    _ma = RemoteMasterArm(
        host       = ROBOT_PC_IP,
        state_port = MA_STATE_PORT,
        cmd_port   = MA_CMD_PORT,
        timeout    = 5.0,
    )

    if not _ma.connect(verbose=True):
        raise RuntimeError(
            f"master_arm_server에 연결 실패: {ROBOT_PC_IP}:{MA_CMD_PORT}\n"
            f"  Robot PC에서 master_arm_server.py를 실행하세요:\n"
            f"  python master_arm_server.py --state-port {MA_STATE_PORT} --cmd-port {MA_CMD_PORT}"
        )

    # 중력 보상 모드 시작 요청
    if not _ma.start_gravity():
        logger.warning("start_gravity 명령 실패 (서버가 이미 실행 중일 수 있습니다)")

    # 첫 상태 수신 대기 (최대 3초)
    waited = 0.0
    while waited < 3.0:
        q, _, _, _ = _ma.get_state()
        if not _ma.is_stale(max_age_secs=2.0):
            break
        time.sleep(0.1)
        waited += 0.1

    if _ma.is_stale(max_age_secs=2.0):
        logger.warning("Master Arm 상태 수신 지연 — 서버의 상태 스트리밍을 확인하세요.")

    master_arm = _ma
    logger.info(f"✅ RemoteMasterArm 연결 완료 ({ROBOT_PC_IP})")
    return _ma


# ── 실행 ──────────────────────────────────────────────────
master_arm = init_master_arm()

q_init, _, _, _ = master_arm.get_state()
print(f"✅ RemoteMasterArm 연결 완료")
print(f"   서버             : {ROBOT_PC_IP}  (state:{MA_STATE_PORT} / cmd:{MA_CMD_PORT})")
print(f"   오른팔 q (deg)   : {np.round(np.rad2deg(q_init[MASTER_RIGHT_SLICE]), 1)}")
print(f"   왼팔  q (deg)   : {np.round(np.rad2deg(q_init[MASTER_LEFT_SLICE]),  1)}")


---
## Step 3 : 그리퍼 초기화 (선택)

`USE_GRIPPER = False`이면 이 셀을 건너뛰어도 됩니다.  
그리퍼는 **Remote Gripper** (UDP 통신) 방식을 사용합니다.

### ▶ 사전 준비: Robot PC에서 gripper_server.py 실행

그리퍼는 Robot PC의 USB에 연결되어 있어 **Remote PC에서 직접 접근 불가**합니다.  
노트북 실행 전에 Robot PC에서 아래 명령을 실행하세요.

```bash
# Robot PC (UPC) 터미널에서:
cd /path/to/rby1-data-collection
python gripper_server.py --port 5009
```

| 구성 요소 | 실행 위치 | 포트 |
|---|---|---|
| `gripper_server.py` | **Robot PC** (192.168.0.56) | UDP 5009 |
| `remote_gripper.py` | Remote PC (이 노트북) | — |

> 아래 진단 셀로 서버 연결을 먼저 확인하세요.


In [ ]:
# [Step 3-1] gripper_server.py 연결 진단 (네트워크 + UDP ping)
# ══════════════════════════════════════════════════════════
# 🔍 gripper_server.py 연결 진단
# ══════════════════════════════════════════════════════════
import socket as _socket
import subprocess as _subprocess
import json as _json
import time as _time

GRIPPER_HOST = CONFIG.get("remote_gripper_host", "192.168.0.56")
GRIPPER_PORT = int(CONFIG.get("remote_gripper_port", 5009))

def diagnose_gripper_server(host: str, port: int, timeout: float = 3.0):
    print("=" * 60)
    print("  Gripper Server 연결 진단")
    print("=" * 60)
    print(f"\n  서버 주소 : {host}:{port} (UDP)")

    # 1) 네트워크 ping
    try:
        r = _subprocess.run(["ping", "-c", "1", "-W", "1", host],
                            capture_output=True, text=True, timeout=5)
        reachable = r.returncode == 0
        print(f"\n[1] 네트워크 ping    : {'✅ 응답' if reachable else '❌ 응답 없음'}")
    except Exception as e:
        reachable = False
        print(f"\n[1] 네트워크 ping    : ❌ 실패 ({e})")

    # 2) UDP ping
    sock = _socket.socket(_socket.AF_INET, _socket.SOCK_DGRAM)
    sock.settimeout(timeout)
    server_ok = False
    try:
        payload = _json.dumps({"cmd": "ping", "ts": _time.time()}).encode()
        sock.sendto(payload, (host, port))
        raw, _ = sock.recvfrom(65535)
        resp   = _json.loads(raw.decode())
        server_ok = resp.get("ok", False)
        initialized = resp.get("initialized", "?")
        print(f"[2] gripper_server   : {'✅ 응답 OK' if server_ok else f'❌ 응답 이상: {resp}'}")
        print(f"    그리퍼 초기화 여부: {initialized}")
    except _socket.timeout:
        print(f"[2] gripper_server   : ❌ 타임아웃 ({timeout}s) — 서버가 실행 중인지 확인")
    except Exception as e:
        print(f"[2] gripper_server   : ❌ 오류 ({e})")
    finally:
        sock.close()

    print("\n" + "=" * 60)
    if server_ok:
        print("✅ gripper_server 연결 확인 — 아래 init 셀을 실행하세요.")
    else:
        print("⚠️  gripper_server 연결 실패. 아래를 확인하세요:")
        print(f"   1. Robot PC({host})에서 아래 명령 실행:")
        print(f"      python gripper_server.py --port {port}")
        print(f"   2. 방화벽: sudo ufw allow {port}/udp")
        print(f"   3. config.yaml의 remote_gripper_host 값이 올바른지 확인")
        print(f"      현재값: {host}")
    return server_ok


_gripper_server_ok = diagnose_gripper_server(GRIPPER_HOST, GRIPPER_PORT)


In [ ]:
# [Step 3-2] 그리퍼 초기화 (homing + 열기)
def init_gripper() -> Optional[RemoteGripper]:
    global gripper

    if not USE_GRIPPER:
        logger.info("그리퍼 비활성화 (USE_GRIPPER=False)")
        return None

    # 로봇 tool flange 12V 공급
    for arm in ("left", "right"):
        if not robot.set_tool_flange_output_voltage(arm, 12):
            logger.warning(f"[{arm}] tool flange 12V 공급 실패")
    time.sleep(0.5)

    _gripper = RemoteGripper()
    if not _gripper.initialize(verbose=True):
        raise RuntimeError("그리퍼 초기화 실패")

    _gripper.homing()
    _gripper.start()
    _gripper.set_normalized_target(np.array([1.0, 1.0]))   # 완전 열기 (normalized_q=1.0 → OPEN)
    time.sleep(0.3)

    gripper = _gripper
    logger.info("✅ 그리퍼 초기화 완료")
    return _gripper


# ── 실행 ──────────────────────────────────────────────────
gripper = init_gripper()
print(f"✅ 그리퍼 상태: {'활성화' if gripper is not None else '비활성화'}")


---
## Step 4 : 로봇 초기 자세 이동

텔레오퍼레이션을 시작하기 전에 로봇을 안전한 초기 자세로 이동시킵니다.  
이 셀이 완료된 후 **Master Arm을 현재 로봇 자세와 유사한 위치로 맞춰주세요.**


In [ ]:
# [Step 4] Robot init pose via waypoints (collision-safe)
# ══════════════════════════════════════════════════════════
# Waypoint logic matches vr_communicate.py (handle_vr_button_event, Left Y):
#
#   elbows NOT bent (robot at zero)  → midpoint1 → midpoint2 → init
#   elbows bent     (in task pose)   → (skip mp1) midpoint2   → init
#
#   midpoint1: arms raised/spread to clear table before bending
#   midpoint2: arms at neutral shoulder, elbows still bent — always executed
# ══════════════════════════════════════════════════════════
from utils import elbows_bending_check

# ── Waypoint definitions (matches Settings in setup.py) ──
SHOULDER_PITCH = 70.0
SHOULDER_ROLL  = 30.0
ELBOW_ANGLE    = -100.0
WRIST_ANGLE    = -70.0

# midpoint1: shoulder raised + spread, elbows bent (arms "up and out")
RIGHT_ARM_MID1_RAD = np.deg2rad([SHOULDER_PITCH, -SHOULDER_ROLL, 0.0, ELBOW_ANGLE, 0.0, WRIST_ANGLE, 0.0])
LEFT_ARM_MID1_RAD  = np.deg2rad([SHOULDER_PITCH,  SHOULDER_ROLL, 0.0, ELBOW_ANGLE, 0.0, WRIST_ANGLE, 0.0])

# midpoint2: shoulder neutral, elbows bent (arms "at side, bent")
RIGHT_ARM_MID2_RAD = np.deg2rad([0.0, -15.0, 0.0, ELBOW_ANGLE, 0.0, WRIST_ANGLE, 0.0])
LEFT_ARM_MID2_RAD  = np.deg2rad([0.0,  15.0, 0.0, ELBOW_ANGLE, 0.0, WRIST_ANGLE, 0.0])


def _movej(torso, right_arm, left_arm, head, minimum_time=5.0) -> bool:
    """Send a single joint-position command (blocking). Mirrors helper.py movej."""
    body_cmd = rby.BodyComponentBasedCommandBuilder()
    if torso is not None:
        body_cmd.set_torso_command(
            rby.JointPositionCommandBuilder()
            .set_minimum_time(minimum_time).set_position(torso))
    if right_arm is not None:
        body_cmd.set_right_arm_command(
            rby.JointPositionCommandBuilder()
            .set_minimum_time(minimum_time).set_position(right_arm))
    if left_arm is not None:
        body_cmd.set_left_arm_command(
            rby.JointPositionCommandBuilder()
            .set_minimum_time(minimum_time).set_position(left_arm))

    cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_cmd)
    if head is not None:
        try:
            cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_minimum_time(minimum_time).set_position(head))
        except Exception:
            pass

    rv = robot.send_command(rby.RobotCommandBuilder().set_command(cbc), 1).get()
    if rv.finish_code != rby.RobotCommandFeedback.FinishCode.Ok:
        logging.error(f"movej failed: finish_code={rv.finish_code}")
        return False
    return True


def move_to_init_pose(minimum_time: float = 7.0) -> bool:
    """
    Move robot to init pose via waypoints (blocking).

    Matches vr_communicate.py handle_vr_button_event (Left Y / Bimanual mode):

      skip = elbows_bending_check(robot)   # True if elbows ARE bent

      if not skip:          → midpoint1   (only from zero/straight)
      (always)              → midpoint2   (unconditional — always run)
      (always)              → init pose

    Why:
      - From zero, arms need to be raised first (midpoint1) so bending elbows
        doesn't cause the forearm to sweep through the table.
      - From task pose (bent elbows), midpoint1 is skipped — we're already "up".
      - midpoint2 is always executed as a safe intermediate before final init.
    """
    torso_rad = np.deg2rad(TORSO_INIT_DEG)
    right_rad = np.deg2rad(RIGHT_ARM_INIT_DEG)
    left_rad  = np.deg2rad(LEFT_ARM_INIT_DEG)
    head_rad  = np.deg2rad(HEAD_INIT_DEG)
    torso_zeros = np.zeros(len(TORSO_INIT_DEG))

    # ── Elbow state diagnosis ─────────────────────────────
    elbows_bent = elbows_bending_check(robot)   # True = already in task pose

    # Read actual angles for display
    try:
        rs = robot.get_state()
        pos = np.array(rs.position)
        hdof = len(robot_model.head_idx)
        tdof = len(robot_model.torso_idx)
        rdof = len(robot_model.right_arm_idx)
        r_elb = np.rad2deg(pos[hdof + tdof + 3])
        l_elb = np.rad2deg(pos[hdof + tdof + rdof + 3])
        thr   = float(CONFIG.get('elbow_angle_threshold_deg', 90))
        print(f"  📐 Elbow angles  R: {r_elb:+.1f}°  L: {l_elb:+.1f}°  (threshold ±{thr}°)")
    except Exception:
        pass

    if elbows_bent:
        print("  ⚠️  Elbows bent → skip midpoint1, run midpoint2 → init")
    else:
        print("  ✅  Elbows straight → midpoint1 → midpoint2 → init")

    # ── Step 1: midpoint1 — only if elbows NOT bent (from zero) ──
    # (vr_communicate: `if not started and not skip_movej_due_to_elbow`)
    if not elbows_bent:
        logger.info("[InitPose] Step 1/3: midpoint1 (raise arms to clear table)")
        print("  ⏳ Step 1/3: midpoint1 — raising arms...")
        ok = _movej(torso_zeros, RIGHT_ARM_MID1_RAD, LEFT_ARM_MID1_RAD,
                    head_rad, minimum_time=minimum_time)
        if not ok:
            logger.warning("[InitPose] midpoint1 failed (continuing)")

    # ── Step 2: midpoint2 — ALWAYS (unconditional) ───────────────
    # (vr_communicate: no condition on elbow check for midpoint2)
    step = "2" if not elbows_bent else "1"
    total = "3" if not elbows_bent else "2"
    logger.info("[InitPose] Step 2: midpoint2 (shoulder neutral)")
    print(f"  ⏳ Step {step}/{total}: midpoint2 — shoulder neutral...")
    ok = _movej(torso_zeros, RIGHT_ARM_MID2_RAD, LEFT_ARM_MID2_RAD,
                head_rad, minimum_time=minimum_time)
    if not ok:
        logger.warning("[InitPose] midpoint2 failed (continuing)")

    # ── Step 3: final init pose ───────────────────────────────────
    logger.info("[InitPose] Final: init pose")
    print(f"  ⏳ Step {int(step)+1}/{total}: init pose...")
    ok = _movej(torso_rad, right_rad, left_rad, head_rad, minimum_time=minimum_time)
    if ok:
        logger.info("✅ Init pose reached")
    else:
        logger.error("Init pose move failed")
    return ok


# ── Execute ──────────────────────────────────────────────
print("⏳ Moving robot to init pose (via waypoints)...")
ok = move_to_init_pose(minimum_time=7.0)
print(f"{'✅ Done' if ok else '❌ Failed'} — Align Master Arm to robot pose.")

## Step 4.1 — Mobile Base Control (Joystick Mode)

**제어 방식: ToggleButton + 20Hz 가속·감속 루프 (main.py 동일 모델)**

| 버튼 | 방향 | 설명 |
|------|------|------|
| **W ↑** | 전진 | 클릭: 가속 시작 / 다시 클릭: 감속 정지 |
| **S ↓** | 후진 | 〃 |
| **A ↺** | 제자리 좌회전 (CCW) | 〃 |
| **D ↻** | 제자리 우회전 (CW) | 〃 |
| **Q ←** | 왼쪽 측면 이동 | 〃 |
| **E →** | 오른쪽 측면 이동 | 〃 |
| **■ STOP** | 즉시 정지 | 속도·입력 모두 0 으로 강제 리셋 |

> 복수 버튼을 동시에 켜면 방향이 합산됩니다 (예: W + D → 전진하면서 우회전).


In [ ]:
# [Step 4.1] Mobile Base Control — Joystick Mode
# ══════════════════════════════════════════════════════════════════════
# main.py 방식: create_command_stream() + 가속·감속 모델 (20Hz 루프)
# ToggleButton: 클릭 → 가속 시작 / 다시 클릭 → 감속 후 정지
# ══════════════════════════════════════════════════════════════════════
import ipywidgets as widgets
import threading
import time
import numpy as np
from IPython.display import display

# ── 파라미터 (main.py Settings 와 동일한 구조) ─────────────────────────
_DT         = 0.05    # 제어 루프 주기 (s) — 20 Hz
_HOLD_MULT  = 10      # hold_time = _DT * _HOLD_MULT

_MAX_LIN    = 0.1    # m/s
_MAX_ANG    = 0.1    # rad/s

# 가속도 (per tick): terminal_v = accel / damp × max_input
# lin: 0.03 / 0.15 = 0.20 m/s  ✓   ang: 0.075 / 0.15 = 0.50 rad/s  ✓
_LIN_ACCEL  = 0.03    # main.py mobile_linear_acceleration_gain
_ANG_ACCEL  = 0.03   # main.py mobile_angular_acceleration_gain
_LIN_DAMP   = 0.15    # main.py mobile_linear_damping_gain
_ANG_DAMP   = 0.15    # main.py mobile_angular_damping_gain

# ── 상태 컨텍스트 (global 재할당 없이 dict 사용) ─────────────────────────
_ctx = {
    "stream": None,
    "vel":    np.zeros(3),    # [vx, vy, wz]  현재 속도
    "target": np.zeros(3),    # [vx, vy, wz]  방향 입력 (-1 / 0 / +1)
    "stop_ev": threading.Event(),
    "thread":  None,
}


def _ensure_stream():
    """스트림이 없으면 생성. main.py: robot.create_command_stream() 패턴."""
    if _ctx["stream"] is not None:
        return True
    try:
        if robot.wait_for_control_ready(3):
            _ctx["stream"] = robot.create_command_stream()
            return True
        _base_status.value = "<b style='color:orange'>⚠️ wait_for_control_ready timeout</b>"
    except Exception as e:
        _base_status.value = f"<b style='color:red'>❌ stream error: {e}</b>"
    return False


def _control_loop():
    """20Hz 제어 루프 — main.py 메인 루프의 가속·감속 모델 그대로 구현."""
    while not _ctx["stop_ev"].is_set():
        t0 = time.perf_counter()

        tgt = _ctx["target"].copy()   # [vx_dir, vy_dir, wz_dir]
        v   = _ctx["vel"]             # 현재 속도 (in-place 수정)

        # ── 가속 + 감속 (main.py 와 동일) ─────────────────────────────
        # v += accel_gain * input
        # v -= damp_gain  * v     (입력 없을 때 지수 감쇠)
        for i in range(2):   # vx, vy (linear)
            v[i] += _LIN_ACCEL * tgt[i]
            if abs(tgt[i]) < 0.01:
                v[i] *= (1.0 - _LIN_DAMP)
        v[2] += _ANG_ACCEL * tgt[2]  # wz (angular)
        if abs(tgt[2]) < 0.01:
            v[2] *= (1.0 - _ANG_DAMP)

        # ── 클램프 ───────────────────────────────────────────────────
        v[0] = float(np.clip(v[0], -_MAX_LIN, _MAX_LIN))
        v[1] = float(np.clip(v[1], -_MAX_LIN, _MAX_LIN))
        v[2] = float(np.clip(v[2], -_MAX_ANG, _MAX_ANG))

        # ── 극소값 제거 ──────────────────────────────────────────────
        v[np.abs(v) < 5e-4] = 0.0

        # ── 명령 전송 (움직임이 있거나 입력 중일 때만) ───────────────
        if np.any(v != 0.0) or np.any(tgt != 0.0):
            if _ensure_stream():
                hold = _DT * _HOLD_MULT
                try:
                    _ctx["stream"].send_command(
                        rby.RobotCommandBuilder().set_command(
                            rby.ComponentBasedCommandBuilder().set_mobility_command(
                                rby.SE2VelocityCommandBuilder()
                                .set_command_header(
                                    rby.CommandHeaderBuilder().set_control_hold_time(hold)
                                )
                                .set_velocity([v[0], v[1]], v[2])
                                .set_minimum_time(_DT * 1.01)
                            )
                        )
                    )
                except Exception as ex:
                    _ctx["stream"] = None   # 다음 루프에서 재생성

        # ── 속도 표시 업데이트 (0.1s 마다) ──────────────────────────
        # (thread에서 widget 업데이트 — IPython 커널은 thread-safe)
        _vel_bar.value = (
            f"<tt style='font-size:11px'>"
            f"vx: <b>{v[0]:+.3f}</b> m/s &nbsp;"
            f"vy: <b>{v[1]:+.3f}</b> m/s &nbsp;"
            f"wz: <b>{v[2]:+.3f}</b> rad/s</tt>"
        )

        elapsed = time.perf_counter() - t0
        time.sleep(max(0.0, _DT - elapsed))


def _start_loop():
    """제어 루프 스레드 시작 (이미 실행 중이면 무시)."""
    if _ctx["thread"] is not None and _ctx["thread"].is_alive():
        return
    _ctx["stop_ev"].clear()
    _ctx["thread"] = threading.Thread(target=_control_loop, daemon=True, name="base_ctrl")
    _ctx["thread"].start()


# ── ToggleButton 위젯 ──────────────────────────────────────────────────
_TBL  = widgets.Layout(width="72px", height="52px")
_TBLS = widgets.Layout(width="90px", height="52px")

_tb_fwd  = widgets.ToggleButton(description="W ↑",    layout=_TBL, button_style="primary")
_tb_back = widgets.ToggleButton(description="S ↓",    layout=_TBL, button_style="primary")
_tb_rotL = widgets.ToggleButton(description="A ↺",    layout=_TBL, button_style="info")
_tb_rotR = widgets.ToggleButton(description="D ↻",    layout=_TBL, button_style="info")
_tb_strL = widgets.ToggleButton(description="Q ←",    layout=_TBL, button_style="")
_tb_strR = widgets.ToggleButton(description="E →",    layout=_TBL, button_style="")
_b_stop  = widgets.Button(      description="■ STOP", layout=_TBLS, button_style="danger")

_base_status = widgets.HTML(value="<b>🟢 Ready — 방향 버튼을 누르면 가속 시작</b>")
_vel_bar     = widgets.HTML(value="<tt style='font-size:11px'>vx: 0.000  vy: 0.000  wz: 0.000</tt>")


def _sync_target():
    """ToggleButton 상태 → 방향 입력 벡터 갱신."""
    _ctx["target"][:] = [
        (1.0 if _tb_fwd.value  else 0.0) - (1.0 if _tb_back.value else 0.0),  # vx
        (1.0 if _tb_strL.value else 0.0) - (1.0 if _tb_strR.value else 0.0),  # vy
        (1.0 if _tb_rotL.value else 0.0) - (1.0 if _tb_rotR.value else 0.0),  # wz
    ]


def _on_toggle(change):
    _sync_target()
    active = []
    if _tb_fwd.value:  active.append("↑Fwd")
    if _tb_back.value: active.append("↓Back")
    if _tb_rotL.value: active.append("↺RotL")
    if _tb_rotR.value: active.append("↻RotR")
    if _tb_strL.value: active.append("←StrafeL")
    if _tb_strR.value: active.append("→StrafeR")
    if active:
        _base_status.value = f"<b style='color:royalblue'>▶ {' + '.join(active)} — 가속 중</b>"
    else:
        _base_status.value = "<b style='color:gray'>🟡 감속 중...</b>"


def _on_stop(b):
    """STOP: 모든 토글 해제 + 속도 즉시 0."""
    for tb in [_tb_fwd, _tb_back, _tb_rotL, _tb_rotR, _tb_strL, _tb_strR]:
        tb.value = False
    _ctx["target"][:] = 0.0
    _ctx["vel"][:] = 0.0
    _base_status.value = "<b style='color:red'>■ 즉시 정지</b>"


for _tb in [_tb_fwd, _tb_back, _tb_rotL, _tb_rotR, _tb_strL, _tb_strR]:
    _tb.observe(_on_toggle, names="value")
_b_stop.on_click(_on_stop)

# ── UI 레이아웃 ────────────────────────────────────────────────────────
_gap        = widgets.Box(layout=widgets.Layout(width="72px"))
_row_fwd    = widgets.HBox([_gap, _tb_fwd, _gap])
_row_mid    = widgets.HBox([_tb_rotL, _b_stop, _tb_rotR])
_row_back   = widgets.HBox([_gap, _tb_back, _gap])
_row_strafe = widgets.HBox([_tb_strL, _tb_strR])

_base_ui = widgets.VBox([
    widgets.HTML("<h4 style='margin:4px 0'>🎮 Mobile Base — Joystick Mode</h4>"),
    widgets.HTML(
        f"<small>Max: {_MAX_LIN} m/s / {_MAX_ANG} rad/s &nbsp;·&nbsp; "
        f"Accel: {_LIN_ACCEL}/{_ANG_ACCEL} per tick &nbsp;·&nbsp; "
        f"Damp: {_LIN_DAMP} &nbsp;·&nbsp; DT: {_DT}s (20 Hz)</small>"
    ),
    widgets.HTML(
        "<small style='color:#555'>"
        "🔵 버튼 클릭 → 가속 시작 &nbsp;|&nbsp; 다시 클릭 → 감속 정지 &nbsp;|&nbsp; "
        "🔴 STOP → 즉시 정지</small>"
    ),
    _row_fwd, _row_mid, _row_back, _row_strafe,
    _base_status,
    _vel_bar,
], layout=widgets.Layout(padding="10px", border="1px solid #aaa", width="300px"))

_start_loop()
display(_base_ui)
print(f"✅ Joystick mode ready  (DT={_DT}s · max_lin={_MAX_LIN}m/s · max_ang={_MAX_ANG}rad/s)")
print(f"   가속도: lin={_LIN_ACCEL}/tick  ang={_ANG_ACCEL}/tick  감속: {_LIN_DAMP}/tick")
print("   ToggleButton: 켜짐(가속 시작) → 꺼짐(감속 후 자동 정지)")


---
## Step 4.5 : Master Arm 초기 자세 자동 이동

Robot PC의 `master_arm_server.py`에 homing 명령을 전송합니다.

### 동작 방식
1. **Remote PC** → `homing` 명령 전송 (`RIGHT_ARM_INIT_DEG` / `LEFT_ARM_INIT_DEG`)
2. **Robot PC** → `CurrentBasedPositionControlMode` + 속도 제한(보간)으로 천천히 이동
3. **Robot PC** → 모든 관절 오차 < `threshold_deg` 도달 시 **위치 유지(Hold) 모드**로 전환
   - `CurrentBasedPositionControlMode`로 목표 자세를 능동적으로 고정 유지
   - ⚠️ gravity 모드가 **아님** — 팔을 손으로 움직이려 해도 저항합니다
4. **Step 6 실행** → `start_gravity()` 호출 시 `CurrentControlMode` (중력 보상) 모드로 전환
   - 이 시점부터 팔을 자유롭게 움직일 수 있습니다

### 모드 전환 요약
```
homing 명령 → [이동 중: CurrentBasedPositionControlMode]
                       ↓ 오차 < threshold_deg
              [Hold 모드: CurrentBasedPositionControlMode @ target]  ← 이 셀 완료 시
                       ↓ Step 6 실행 (start_gravity)
              [Gravity 모드: CurrentControlMode + 중력 보상]          ← 텔레오퍼레이션 시작
```

> ⚠️ **주의**: 이 셀을 실행하면 **Master Arm이 자동으로 움직입니다.**  
> 조작자 손을 Master Arm에서 떼고 주위를 정리한 후 실행하세요.


In [ ]:
# [Step 4.5] Master Arm 초기 자세 자동 이동 (homing 명령 전송)
def move_master_arm_to_init_pose(
    timeout: float = 15.0,
    threshold_deg: float = 5.0,
    torque_limit: Optional[list] = None,
    max_speed_deg_per_sec: float = 10.0,
) -> bool:
    """
    master_arm_server.py에 homing 명령을 전송하여
    Robot PC의 Master Arm을 초기 자세로 자동 이동시킵니다.

    Args:
        timeout            : 완료 대기 최대 시간 (초)
        threshold_deg      : 수렴 판정 오차 (도)
        torque_limit       : 관절별 전류 한계 (None = 기본값 사용)
        max_speed_deg_per_sec: 각 관절의 최대 이동 속도 (deg/sec).
                               낮을수록 천천히 움직입니다. (기본 30 deg/sec)
    """
    if master_arm is None:
        raise RuntimeError("RemoteMasterArm이 초기화되어 있지 않습니다. (Step 2 실행 필요)")

    if torque_limit is None:
        torque_limit = [3.0, 3.0, 3.0, 1.5, 1.5, 1.5, 1.5] * 2

    logger.info(
        f"[MasterArm Homing] 목표 자세 전송\n"
        f"  오른팔: {RIGHT_ARM_INIT_DEG}\n"
        f"  왼팔:   {LEFT_ARM_INIT_DEG}\n"
        f"  최대 속도: {max_speed_deg_per_sec} deg/sec"
    )

    # homing 명령 전송 (비동기 — 서버가 백그라운드에서 실행)
    ok = master_arm.homing(
        target_right_deg      = RIGHT_ARM_INIT_DEG,
        target_left_deg       = LEFT_ARM_INIT_DEG,
        torque_limit          = torque_limit,
        threshold_deg         = threshold_deg,
        max_speed_deg_per_sec = max_speed_deg_per_sec,
    )
    if not ok:
        raise RuntimeError("homing 명령 전송 실패 — 서버 연결을 확인하세요.")

    # 완료 대기
    print(f"⏳ Homing 중... (최대 {timeout}초, 최대 속도 {max_speed_deg_per_sec} deg/sec)")
    reached = master_arm.wait_homing(timeout=timeout)

    # 최종 상태 확인
    q_final, _, _, _ = master_arm.get_state()
    target_q = np.concatenate([
        np.deg2rad(RIGHT_ARM_INIT_DEG),
        np.deg2rad(LEFT_ARM_INIT_DEG),
    ])
    err_deg = np.rad2deg(np.abs(q_final - target_q))

    if reached:
        print(f"✅ Homing 수렴 완료")
    else:
        print(f"⚠️  {timeout}s 타임아웃 — 가능한 범위까지 이동")
    print(f"   최대 관절 오차  : {err_deg.max():.2f}°")
    print(f"   오른팔 오차 (7) : {np.round(err_deg[:7], 1)}°")
    print(f"   왼팔  오차 (7)  : {np.round(err_deg[7:], 1)}°")

    return reached


# ── 실행 ──────────────────────────────────────────────────
# max_speed_deg_per_sec 조절:
#   빠르게 →  60.0 deg/sec
#   기본값 →  30.0 deg/sec  (권장)
#   느리게 →  15.0 deg/sec
print("⚠️  Master Arm이 자동으로 움직입니다. 손을 뗀 후 실행하세요.\n")
reached = move_master_arm_to_init_pose(
    timeout=15.0,
    threshold_deg=5.0,
    max_speed_deg_per_sec=30.0,   # ← 이 값을 조절하여 속도를 변경하세요
)
print()
if reached:
    print("✅ Step 6을 실행하여 텔레오퍼레이션을 시작하세요.")
else:
    print("⚠️  완전히 수렴하지 못했습니다. 수동으로 자세를 확인한 후 Step 6을 실행하세요.")


---
## Step 5 : 카메라 초기화

RealSense 카메라들을 시작합니다. 설정 파일의 `cameras` 섹션에서 시리얼 번호를 읽어옵니다.  
카메라가 없으면 `cameras: {}` 로 비워두고 진행할 수 있습니다.


In [ ]:
# [Step 5-1] RealSense 카메라 초기화 및 파이프라인 시작
import gc
import pyrealsense2 as _prs


def _hardware_reset_cameras(target_serials: set) -> None:
    """대상 시리얼의 카메라를 하드웨어 리셋하여 USB 상태를 초기화합니다."""
    ctx = _prs.context()
    devices = ctx.query_devices()
    reset_count = 0
    for dev in devices:
        sn = dev.get_info(_prs.camera_info.serial_number)
        if sn in target_serials:
            try:
                dev.hardware_reset()
                print(f"  ✅ {sn} 리셋 완료")
                reset_count += 1
            except Exception as e:
                print(f"  ⚠️ {sn} 리셋 실패: {e}")
    del devices, dev, ctx
    gc.collect()
    if reset_count > 0:
        print(f"  ⏳ USB 재열거 대기 중 (5초)...")
        time.sleep(5.0)


def _query_available_cameras() -> dict:
    """연결된 RealSense 장치를 조회합니다. (컨텍스트를 즉시 해제)"""
    ctx = _prs.context()
    result = {}
    for dev in ctx.query_devices():
        result[dev.get_info(_prs.camera_info.serial_number)] = dev.get_info(_prs.camera_info.name)
    del ctx
    gc.collect()
    return result


def init_cameras() -> Optional[MultiRealsense]:
    global realsense, PRIMARY_SERIAL

    if not CAM_NAMES:
        logger.warning("cameras 설정 없음 — 카메라 없이 진행합니다.")
        return None

    # 기존 카메라 리소스 먼저 해제 (재실행 시 'device busy' 방지)
    stop_cameras()
    gc.collect()
    time.sleep(2.0)  # 장치 해제 후 안정화 대기

    # ── 사전 진단: 연결된 RealSense 장치 확인 ─────────────────
    _available = _query_available_cameras()

    print("─" * 60)
    print(f"  [카메라 진단] 연결된 RealSense 장치: {len(_available)}개")
    for serial, name in _available.items():
        cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "미설정")
        mark = "✅" if serial in CAM_SERIALS.values() else "ℹ️ config 없음"
        print(f"  {mark}  {cfg_name:>6}  |  {name}  |  S/N: {serial}")

    _not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
    if _not_found:
        print(f"\n  ❌ config에 있지만 감지 안 된 카메라:")
        for n, s in _not_found:
            print(f"      {n} ({s})  → USB 케이블/포트를 확인하세요")
    else:
        print(f"\n  ✅ config의 모든 카메라 ({len(CAM_SERIALS)}개)가 감지됨")
    print("─" * 60)

    if not _available:
        raise RuntimeError("RealSense 카메라가 하나도 감지되지 않습니다. USB 연결을 확인하세요.")

    # ── 하드웨어 리셋: USB/UVC 상태 완전 초기화 ───────────────
    _target_serials = set(CAM_SERIALS.values()) & set(_available.keys())
    print("\n🔄 모든 카메라 하드웨어 리셋 중...")
    _hardware_reset_cameras(_target_serials)
    print("✅ 하드웨어 리셋 후 USB 재열거 대기 완료\n")

    # ── 카메라 파이프라인 시작 (최대 2회 전체 재시도) ─────────
    serials = [CAM_SERIALS[n] for n in CAM_NAMES]
    max_init_attempts = 2
    _rs = None

    for attempt in range(1, max_init_attempts + 1):
        if attempt > 1:
            print(f"\n🔄 카메라 전체 초기화 재시도 ({attempt}/{max_init_attempts})...")
            gc.collect()
            time.sleep(3.0)

        _rs = MultiRealsense(
            camera_serials=serials,
            width=IMG_WIDTH,
            height=IMG_HEIGHT,
        )
        _rs.start()

        if _rs.running:
            break
        else:
            logger.warning(f"카메라 초기화 시도 {attempt}/{max_init_attempts} 실패")
            try:
                _rs.stop()
            except Exception:
                pass
            _rs = None

    # ── 시작 결과 확인 ────────────────────────────────────────
    if _rs is None or not _rs.running:
        raise RuntimeError(
            "카메라 파이프라인 시작 실패 — 모든 카메라가 응답하지 않습니다.\n"
            "  • 카메라 USB 연결 상태를 확인하세요.\n"
            "  • 다른 프로세스가 카메라를 점유 중인지 확인 (realsense-viewer 등).\n"
            "  • USB 허브가 아닌 PC에 직접 연결하세요.\n"
            "  • 각 카메라를 별도 USB 컨트롤러(버스)에 연결하세요."
        )

    started = list(_rs.buffers.keys())
    failed  = [s for s in serials if s not in started]
    if failed:
        failed_names = [n for n, s in CAM_SERIALS.items() if s in failed]
        logger.warning(
            f"카메라 시작 실패 ({failed_names}): {failed}\n"
            f"  시작된 카메라만 사용합니다: {[n for n in CAM_NAMES if CAM_SERIALS[n] in started]}"
        )

    if not started:
        raise RuntimeError("시작된 카메라가 없습니다.")

    # ── PRIMARY_SERIAL 갱신 (head cam 등 주 카메라가 실패한 경우) ──
    if PRIMARY_SERIAL not in started:
        old_name = next((n for n, s in CAM_SERIALS.items() if s == PRIMARY_SERIAL), PRIMARY_SERIAL)
        PRIMARY_SERIAL = started[0]
        new_name = next((n for n, s in CAM_SERIALS.items() if s == PRIMARY_SERIAL), PRIMARY_SERIAL)
        logger.warning(f"주 카메라({old_name}) 시작 실패 → {new_name}(으)로 자동 변경됨")

    shared_state.camera_warmup_done = False
    realsense = _rs

    # 첫 프레임 수신 대기 (최대 10초 — 하드웨어 리셋 후 안정화 포함)
    waited = 0.0
    while waited < 10.0:
        if _rs.get_frames():
            break
        time.sleep(0.2)
        waited += 0.2
    if waited >= 10.0:
        logger.warning("카메라 첫 프레임 수신 타임아웃 (10s) — 실행은 계속합니다.")

    started_names = [n for n in CAM_NAMES if CAM_SERIALS[n] in started]
    logger.info(f"✅ 카메라 스트림 시작: {started_names}")
    return _rs


def stop_cameras():
    global realsense
    if realsense is not None:
        try:
            realsense.stop()
        except Exception as e:
            logger.warning(f"카메라 정지 중 오류: {e}")
        realsense = None
        gc.collect()


# ── 실행 ──────────────────────────────────────────────────
realsense = init_cameras()
if realsense:
    started_names = [n for n in CAM_NAMES if CAM_SERIALS[n] in realsense.buffers]
    print(f"\n✅ 카메라 초기화 완료: {', '.join(started_names)}")
    print(f"   주 카메라 (PRIMARY_SERIAL): {next((n for n, s in CAM_SERIALS.items() if s == PRIMARY_SERIAL), PRIMARY_SERIAL)}")
else:
    print("✅ 카메라 초기화: 비활성화 (카메라 없음)")

In [ ]:
# [Step 5-2] Camera initial frame check & visualization
# ── Initial frame check ────────────────────────────────────────────────────
# Visualize RGB + Depth images for each camera.
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

if realsense is None:
    print("Camera is disabled.")
else:
    # retry up to 5 seconds until frames are ready
    frames = {}
    for _ in range(50):
        frames = realsense.get_frames()
        if frames:
            break
        time.sleep(0.1)

    if not frames:
        print("No frames received. Check camera status.")
    else:
        # serial → name reverse mapping
        serial_to_name = {s: n for n, s in CAM_SERIALS.items()}

        n_cams = len(frames)
        fig, axes = plt.subplots(n_cams, 2, figsize=(12, 4 * n_cams))
        if n_cams == 1:
            axes = [axes]  # normalize to array for single camera

        for row_idx, (serial, frame) in enumerate(sorted(frames.items())):
            cam_name = serial_to_name.get(serial, serial)

            # ── Color (BGR → RGB) ──────────────────────────────
            color_rgb = frame.color[:, :, ::-1]
            axes[row_idx][0].imshow(color_rgb)
            axes[row_idx][0].set_title(f"[{cam_name}]  Color  (S/N: {serial})", fontsize=11)
            axes[row_idx][0].axis("off")

            # ── Depth (mm → colormap) ────────────────────────
            depth_m = frame.depth.astype(float) / 1000.0  # mm → m
            depth_m[depth_m == 0] = np.nan                # mask invalid pixels

            vmin = np.nanpercentile(depth_m, 2)
            vmax = np.nanpercentile(depth_m, 98)
            im = axes[row_idx][1].imshow(depth_m, cmap="turbo", vmin=vmin, vmax=vmax)
            axes[row_idx][1].set_title(f"[{cam_name}]  Depth (m)  (S/N: {serial})", fontsize=11)
            axes[row_idx][1].axis("off")
            plt.colorbar(im, ax=axes[row_idx][1], fraction=0.046, pad=0.04)

            ts_str = f"t = {frame.t:.3f} s"
            for ax in axes[row_idx]:
                ax.text(0.01, 0.01, ts_str, transform=ax.transAxes,
                        fontsize=8, color="white",
                        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5))

        fig.suptitle(
            f"Initial Frames ({n_cams} cameras)  |  {frames[list(frames.keys())[0]].color.shape[1]}x{frames[list(frames.keys())[0]].color.shape[0]}",
            fontsize=13, y=1.01
        )
        plt.tight_layout()
        plt.show()
        print(f"Frame check OK: {[serial_to_name.get(s, s) for s in frames]}")


---
## Step 6 : 텔레오퍼레이션 시작

**SDK `17_teleoperation_with_joint_mapping` 최신 패턴** 그대로 구현된 텔레오퍼레이션입니다.

### 버튼 / 트리거 동작

| 입력 | 동작 |
|---|---|
| 🔓 **잠금 해제 버튼 누름** (`button`) | Master Arm 팔로우 (오른팔 / 왼팔 독립 제어) |
| 🔒 **잠금 해제 버튼 해제** | 해제 순간의 로봇 자세를 hold (`control_hold_time=1e6`) |
| 🖐 **트리거** (`trigger`, 아날로그) | 쥔 정도에 비례하여 그리퍼 개폐 (0=열림 ~ 1=닫힘) |

### 그리퍼 트리거 정규화 (SDK 최신 패턴)

각 트리거의 고유 raw 범위를 사용하여 [0, 1]로 정규화합니다:

| 트리거 | raw 범위 | 정규화 공식 |
|---|---|---|
| 오른쪽 | 4242 ~ 5335 | `(trigger - 4242) / (5335 - 4242)` |
| 왼쪽 | 7525 ~ 8600 | `(trigger - 7525) / (8600 - 7525)` |

### 동작 방식 (SDK 원본 패턴)

1. **버튼 누름**: 해당 팔에 대해 Master Arm 관절값을 로봇에 전송
   - `minimum_time`: 0.8s에서 시작 → 매 스텝 0.01s씩 감소 → 최소 `CONTROL_DT*1.01`
   - `control_hold_time = 1e6`: 명령 끊어져도 마지막 자세 유지
2. **버튼 해제**: 해당 팔 명령을 **보내지 않음** (SDK 원본 패턴)
   - `control_hold_time=1e6` 덕분에 로봇이 마지막 자세를 자동 유지
   - `minimum_time` 0.8로 리셋

### Master Arm 서버 중력 보상 (SDK 원본 패턴)

```
torque = gravity_term 
       + q_limit_barrier * (barrier) 
       + viscous_gain * qvel
```
- 점성 감쇠(`viscous_gain`)로 팔 안정성 확보 → 튀는 현상 방지
- 관절 한계 배리어로 안전한 범위 내 동작

In [ ]:
# [Step 6] 텔레오퍼레이션 시작 (MA 폴링 + 로봇 명령 스트리밍 스레드)
# ══════════════════════════════════════════════════════════
# ⚙️  Teleop 파라미터  (SDK 17_teleoperation_with_joint_mapping 패턴)
# ══════════════════════════════════════════════════════════

# 그리퍼: 트리거 아날로그 범위 (SDK 최신 패턴)
# 각 트리거는 고유한 raw 범위를 가지며, 이를 [0, 1]로 정규화합니다.
R_TRIGGER_MIN = 4242.0
R_TRIGGER_MAX = 5335.0
L_TRIGGER_MIN = 7525.0
L_TRIGGER_MAX = 8600.0

# minimum_time 초기값 / 하한 (SDK 패턴)
INITIAL_MTIME = 0.8            # 버튼 해제 시 리셋 값
MIN_MTIME     = CONTROL_DT * 1.01  # 하한


# ══════════════════════════════════════════════════════════
# 📡  Master Arm 상태 폴링 스레드
# ══════════════════════════════════════════════════════════

def state_poll_thread_fn(state: SharedState, _ma: RemoteMasterArm,
                         _gripper: Optional[RemoteGripper]):
    """
    100Hz로 Master Arm 상태를 폴링하여 SharedState에 반영.

    - 트리거 → 그리퍼 정규화: per-arm raw 범위 → [0, 1]
    - 잠금 해제 버튼 에지 감지: 상승 에지 → 팔로우, 하강 에지 → hold
    """
    logger.info("Master Arm 상태 폴링 스레드 시작")

    last_grip_t      = 0.0
    last_grip_target = np.array([-1.0, -1.0])  # sentinel (초기 전송 강제)

    # 그리퍼 상태 폴링 주기 카운터 (~30Hz = 100Hz 루프의 3번에 1번)
    _grip_state_tick     = 0
    _GRIP_STATE_INTERVAL = max(1, int(round(1.0 / (30.0 * CONTROL_DT))))  # ≈ 3

    while state.is_teleop_active:
        q, grav, r_trig, l_trig = _ma.get_state()
        r_unlock, l_unlock = _ma.get_unlock_state()
        now = time.time()

        # ── 그리퍼: per-arm raw 범위 → [0, 1] 정규화 ──────────
        r_norm = float(np.clip((r_trig - R_TRIGGER_MIN) / (R_TRIGGER_MAX - R_TRIGGER_MIN), 0.0, 1.0))
        l_norm = float(np.clip((l_trig - L_TRIGGER_MIN) / (L_TRIGGER_MAX - L_TRIGGER_MIN), 0.0, 1.0))
        grip_target = np.array([r_norm, l_norm])

        with state.lock:
            state.master_arm_q       = q
            state.master_arm_gravity = grav
            state.gripper_target[:]  = grip_target

            # ── 잠금 해제 버튼 에지 처리 ─────────────────────────
            r_rising  = bool(r_unlock) and not state.unlock_right_prev
            r_falling = not bool(r_unlock) and state.unlock_right_prev
            l_rising  = bool(l_unlock) and not state.unlock_left_prev
            l_falling = not bool(l_unlock) and state.unlock_left_prev

            if r_rising:
                state.unlock_right_active = True
                logger.info("[잠금해제↑] 오른팔 팔로우 시작")
            if r_falling:
                rq = state.robot_joint_positions
                if rq.size > 0 and robot_model is not None:
                    state.hold_right_q = rq[list(robot_model.right_arm_idx)].copy()
                state.unlock_right_active = False
                logger.info("[잠금해제↓] 오른팔 위치 고정")
            state.unlock_right_prev = bool(r_unlock)

            if l_rising:
                state.unlock_left_active = True
                logger.info("[잠금해제↑] 왼팔 팔로우 시작")
            if l_falling:
                lq = state.robot_joint_positions
                if lq.size > 0 and robot_model is not None:
                    state.hold_left_q = lq[list(robot_model.left_arm_idx)].copy()
                state.unlock_left_active = False
                logger.info("[잠금해제↓] 왼팔 위치 고정")
            state.unlock_left_prev = bool(l_unlock)

        # ── 그리퍼 타겟 UDP 전송 (변화 ≥2% 또는 0.1초 경과 시) ─
        if _gripper is not None:
            diff = float(np.max(np.abs(grip_target - last_grip_target)))
            if diff >= 0.02 or (now - last_grip_t) >= 0.1:
                try:
                    _gripper.set_normalized_target(grip_target)
                    last_grip_t      = now
                    last_grip_target = grip_target.copy()
                except Exception as e:
                    logger.warning(f"[그리퍼 타겟] 전송 오류: {e}")

        # ── 그리퍼 실제 상태 폴링 (~30Hz) ────────────────────────
        # 데이터 로거가 직접 UDP 호출하면 set_normalized_target ACK와
        # 소켓 경쟁이 발생하므로 여기서 캐시하고 로거는 캐시를 읽습니다.
        # 규칙: get_state()=0=OPEN,1=CLOSED → 1-x 반전 → 0=CLOSED,1=OPEN
        _grip_state_tick += 1
        if _gripper is not None and (_grip_state_tick % _GRIP_STATE_INTERVAL == 0):
            try:
                raw_state = _gripper.get_state()
                if raw_state is not None:
                    inverted = 1.0 - np.asarray(raw_state, dtype=np.float64)
                    with state.lock:
                        state.gripper_state[:] = np.clip(inverted, 0.0, 1.0)
            except Exception as e:
                logger.debug(f"[그리퍼 상태] 읽기 실패 (이전 값 유지): {e}")

        time.sleep(CONTROL_DT)

    logger.info("Master Arm 상태 폴링 스레드 종료")


# ══════════════════════════════════════════════════════════
# 🤖  로봇 명령 스트리밍 스레드
# ══════════════════════════════════════════════════════════

# 로봇 관절 한계 (목표 위치 클램프용)
dyn_model    = _robot.get_dynamics()
dyn_state    = dyn_model.make_state([], robot_model.robot_joint_names)
robot_min_q  = dyn_model.get_limit_q_lower(dyn_state)
robot_max_q  = dyn_model.get_limit_q_upper(dyn_state)
robot_max_qdot  = dyn_model.get_limit_qdot_upper(dyn_state)
robot_max_qddot = dyn_model.get_limit_qddot_upper(dyn_state)


def robot_command_thread_fn(state: SharedState, _robot, _robot_model):
    """
    SDK 17_teleoperation_with_joint_mapping 패턴으로 로봇에 명령 스트리밍.

    - control_hold_time=1e6: 명령 끊어져도 마지막 자세 유지
    - 버튼 누름 시에만 해당 팔 명령 전송, 해제 시 명령 건너뜀
    - minimum_time: 매 스텝 감소 → MIN_MTIME까지
    - 토르소/헤드는 항상 전송
    """
    torso_rad = np.deg2rad(TORSO_INIT_DEG)
    head_rad  = np.deg2rad(HEAD_INIT_DEG)

    # ── 스트림 생성 ─────────────────────────────────────────
    stream = None
    for attempt in range(20):
        if not state.is_teleop_active:
            return
        try:
            stream = _robot.create_command_stream(priority=1)
            logger.info(f"로봇 명령 스트림 생성 완료 (시도 {attempt+1})")
            break
        except Exception as e:
            logger.warning(f"스트림 생성 대기 ({attempt+1}/20): {e}")
            time.sleep(0.5)
    if stream is None:
        logger.error("로봇 명령 스트림 생성 실패 — 명령 스레드 종료")
        return

    def _make_jpc(pos, mtime, v_lim=None, a_lim=None):
        """JointPositionCommand 빌더 헬퍼 (control_hold_time=1e6 고정)."""
        builder = (
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(1e6))
            .set_position(pos)
            .set_minimum_time(mtime)
        )
        if v_lim is not None:
            builder.set_velocity_limit(v_lim)
        if a_lim is not None:
            builder.set_acceleration_limit(a_lim)
        return builder

    # ── 초기 hold 명령: 현재 로봇 자세 유지 ──────────────────
    with state.lock:
        rq   = state.robot_joint_positions
        ridx = list(_robot_model.right_arm_idx)
        lidx = list(_robot_model.left_arm_idx)
        init_r = rq[ridx].copy() if rq.size > 0 else np.deg2rad(RIGHT_ARM_INIT_DEG)
        init_l = rq[lidx].copy() if rq.size > 0 else np.deg2rad(LEFT_ARM_INIT_DEG)
    try:
        rc_init = (
            rby.BodyComponentBasedCommandBuilder()
            .set_torso_command(_make_jpc(torso_rad, 0.5))
            .set_right_arm_command(_make_jpc(init_r, 0.5))
            .set_left_arm_command(_make_jpc(init_l, 0.5))
        )
        cbc_init = rby.ComponentBasedCommandBuilder().set_body_command(rc_init)
        stream.send_command(rby.RobotCommandBuilder().set_command(cbc_init))
        logger.info("초기 hold 명령 전송 완료")
    except Exception as e:
        logger.warning(f"초기 hold 명령 실패 (계속 진행): {e}")

    logger.info("로봇 명령 스레드 시작")

    right_mtime = INITIAL_MTIME
    left_mtime  = INITIAL_MTIME

    while state.is_teleop_active:
        with state.lock:
            q        = state.master_arm_q
            r_active = state.unlock_right_active
            l_active = state.unlock_left_active

        if q is None or len(q) < 14:
            time.sleep(CONTROL_DT)
            continue

        right_idx = list(_robot_model.right_arm_idx)
        left_idx  = list(_robot_model.left_arm_idx)

        rc = rby.BodyComponentBasedCommandBuilder()
        rc.set_torso_command(_make_jpc(torso_rad, MIN_MTIME))

        # ── 오른팔 ──────────────────────────────────────────
        if r_active:
            right_mtime = max(right_mtime - CONTROL_DT, MIN_MTIME)
            target_r = np.clip(q[MASTER_RIGHT_SLICE], robot_min_q[right_idx], robot_max_q[right_idx])
            rc.set_right_arm_command(
                _make_jpc(target_r, right_mtime,
                          v_lim=robot_max_qdot[right_idx],
                          a_lim=robot_max_qddot[right_idx] * 30)
            )
        else:
            right_mtime = INITIAL_MTIME

        # ── 왼팔 ────────────────────────────────────────────
        if l_active:
            left_mtime = max(left_mtime - CONTROL_DT, MIN_MTIME)
            target_l = np.clip(q[MASTER_LEFT_SLICE], robot_min_q[left_idx], robot_max_q[left_idx])
            rc.set_left_arm_command(
                _make_jpc(target_l, left_mtime,
                          v_lim=robot_max_qdot[left_idx],
                          a_lim=robot_max_qddot[left_idx] * 30)
            )
        else:
            left_mtime = INITIAL_MTIME

        # ── 헤드 + 전송 ─────────────────────────────────────
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(rc)
        try:
            cbc.set_head_command(_make_jpc(head_rad, MIN_MTIME))
        except Exception:
            pass

        try:
            stream.send_command(rby.RobotCommandBuilder().set_command(cbc))
        except Exception as e:
            logger.error(f"명령 전송 오류: {e}")
            try:
                stream = _robot.create_command_stream(priority=1)
                logger.info("스트림 재생성 완료")
            except Exception as e2:
                logger.error(f"스트림 재생성 실패: {e2}")

        time.sleep(CONTROL_DT)

    try:
        stream.cancel()
    except Exception:
        pass
    logger.info("로봇 명령 스레드 종료")


# ══════════════════════════════════════════════════════════
# 🚀  텔레오퍼레이션 시작 / 중단
# ══════════════════════════════════════════════════════════

_state_poll_thread: Optional[threading.Thread] = None
_robot_cmd_thread:  Optional[threading.Thread] = None


def start_teleoperation():
    """텔레오퍼레이션 스레드를 시작합니다 (상태 폴링 + 로봇 명령 스트리밍)."""
    global _state_poll_thread, _robot_cmd_thread

    if shared_state.is_teleop_active:
        logger.warning("이미 텔레오퍼레이션이 활성화되어 있습니다.")
        return

    # Master Arm gravity 모드 활성화
    master_arm.start_gravity()
    time.sleep(0.1)

    with shared_state.lock:
        rq = shared_state.robot_joint_positions
        if rq.size > 0 and robot_model is not None:
            shared_state.hold_right_q = rq[list(robot_model.right_arm_idx)].copy()
            shared_state.hold_left_q  = rq[list(robot_model.left_arm_idx)].copy()
        else:
            shared_state.hold_right_q = np.deg2rad(RIGHT_ARM_INIT_DEG)
            shared_state.hold_left_q  = np.deg2rad(LEFT_ARM_INIT_DEG)

        shared_state.unlock_right_active = False
        shared_state.unlock_left_active  = False
        shared_state.unlock_right_prev   = False
        shared_state.unlock_left_prev    = False
        shared_state.gripper_target[:]   = [0.0, 0.0]

    shared_state.is_teleop_active = True

    _state_poll_thread = threading.Thread(
        target=state_poll_thread_fn,
        args=(shared_state, master_arm, gripper),
        name="ma_poll", daemon=True,
    )
    _state_poll_thread.start()

    # 첫 Master Arm 관절값 수신 대기
    for _ in range(30):
        with shared_state.lock:
            if shared_state.master_arm_q is not None:
                break
        time.sleep(0.05)

    _robot_cmd_thread = threading.Thread(
        target=robot_command_thread_fn,
        args=(shared_state, robot, robot_model),
        name="robot_cmd", daemon=True,
    )
    _robot_cmd_thread.start()
    logger.info("✅ 텔레오퍼레이션 시작")


def stop_teleoperation():
    """텔레오퍼레이션 스레드를 안전하게 중단합니다."""
    global _state_poll_thread, _robot_cmd_thread

    shared_state.is_teleop_active = False

    if _state_poll_thread is not None:
        _state_poll_thread.join(timeout=2.0)
        _state_poll_thread = None

    if _robot_cmd_thread is not None:
        _robot_cmd_thread.join(timeout=2.0)
        _robot_cmd_thread = None

    logger.info("텔레오퍼레이션 중단")


# ── 실행 ──────────────────────────────────────────────────
start_teleoperation()
time.sleep(0.5)

with shared_state.lock:
    q_current = shared_state.master_arm_q

if q_current is not None:
    print("✅ 텔레오퍼레이션 시작 완료")
    print(f"   오른팔 q (deg): {np.round(np.rad2deg(q_current[MASTER_RIGHT_SLICE]), 1)}")
    print(f"   왼팔  q (deg): {np.round(np.rad2deg(q_current[MASTER_LEFT_SLICE]),  1)}")
    print()
    print("🔒 잠금 해제 버튼을 누른 상태에서만 로봇이 팔로우합니다.")
    print("🖐  그리퍼: 트리거를 쥔 정도에 비례하여 개폐됩니다.")
    print(f"    오른쪽 트리거 범위: {R_TRIGGER_MIN:.0f} ~ {R_TRIGGER_MAX:.0f}")
    print(f"    왼쪽  트리거 범위: {L_TRIGGER_MIN:.0f} ~ {L_TRIGGER_MAX:.0f}")
else:
    print("⚠️  Master Arm 관절 값을 아직 수신하지 못했습니다.")

---
## Step 7 : 데이터 수집 루프

에피소드 녹화 → 저장/폐기 → 초기 자세 복귀 → 다음 에피소드 순으로 반복합니다.

### 데모 수집 워크플로우

```
[▶ Start]  →  (데모 수행)  →  [⏹ Stop & Save] 또는 [🗑 Discard]
                                         ↓
                              [🔄 초기 자세 복귀]  ← 로봇 + Master Arm 동시 리셋
                                         ↓
                              [▶ Start]  →  (다음 데모) ...
```

| 버튼 | 단축키 | 동작 |
|---|---|---|
| `▶ Start Recording` | — | 새 에피소드 녹화 시작 |
| `⏹ Stop & Save` | — | 녹화 종료 및 H5 저장 |
| `🗑 Discard` | — | 현재 에피소드 폐기 (파일 삭제) |
| `🔄 초기 자세 복귀` | — | 로봇 + Master Arm을 초기 자세로 이동 후 다음 에피소드 준비 |

> **초기 자세 복귀** 버튼은 녹화 중이어도 자동 저장 후 복귀합니다.  
> 복귀 중에는 모든 버튼이 비활성화되며, 완료 후 자동으로 활성화됩니다.


In [ ]:
# [Step 7-1] 데이터 로거 함수 정의 (H5Writer + 카메라 + 그리퍼 동기 저장)
# ══════════════════════════════════════════════════════════
# 데이터 로거 스레드 (main.py의 start_demo_logger 를 Master Arm 용으로 재구현)
# ══════════════════════════════════════════════════════════

# 그리퍼 상태 기본값 (2차원: [right, left])
_GRIP_DEFAULT = np.zeros(2, dtype=np.float64)

# ── 그리퍼 규칙(convention) 정리 ────────────────────────────────────
#
# [homing 결과 — gripper.py homing()]
#   direction=0 (+토크) → 물리적 OPEN 방향 이동 → min_q = OPEN 엔코더 위치
#   direction=1 (-토크) → 물리적 CLOSED 방향 이동 → max_q = CLOSED 엔코더 위치
#
# [gripper.py get_state()]
#   q_norm = (q - min_q) / (max_q - min_q)
#   → 0 = OPEN  (q ≈ min_q)
#   → 1 = CLOSED (q ≈ max_q)
#
# [gripper.py set_normalized_target(n), GRIPPER_DIRECTION=False]
#   target = (1 - n) × (max_q - min_q) + min_q
#   → n=0 → max_q = CLOSED
#   → n=1 → min_q = OPEN
#   ⇒ 시스템 전반 규칙: normalized_q  0 = CLOSED,  1 = OPEN
#   ⇒ 따라서 초기화 시 set_normalized_target([1, 1]) = 완전 열기 (OPEN)
#
# [gripper_target 저장값 (r_norm)]
#   r_norm=0 (트리거 미입력) → CLOSED
#   r_norm=1 (트리거 완전 입력) → OPEN
#   규칙: 0=CLOSED, 1=OPEN
#
# [get_state() vs gripper_target 비교]
#   get_state():    0=OPEN,   1=CLOSED  ← gripper_target 과 반전!
#   gripper_target: 0=CLOSED, 1=OPEN
#
# [수정] gripper_state 저장 시 1 - get_state() 로 반전
#   OPEN  (get_state=0): 1-0 = 1  → 저장값 1=OPEN  ✓
#   CLOSED(get_state=1): 1-1 = 0  → 저장값 0=CLOSED ✓
#   → gripper_state, gripper_target 모두 0=CLOSED / 1=OPEN 으로 통일
# ────────────────────────────────────────────────────────────────────


def start_data_logger(
    h5_writer: H5Writer,
    state: SharedState,
    _gripper: Optional[RemoteGripper],
    _realsense: Optional[MultiRealsense],
    fps: int = REC_FPS,
) -> threading.Event:
    """
    데이터 수집 루프를 백그라운드 스레드로 시작.

    매 프레임마다 다음을 H5에 기록합니다:
      - robot_position      (현재 로봇 관절 위치)
      - robot_target_joints (Master Arm → 로봇 목표 관절)
      - gripper_state       (실제 그리퍼 위치,  0=CLOSED / 1=OPEN)
      - gripper_target      (명령 목표,          0=CLOSED / 1=OPEN)
      ※ 두 값 모두 동일한 규칙(0=CLOSED, 1=OPEN)을 사용합니다.
      - 카메라 RGB/Depth, PCD 등

    반환된 stop_event.set() 을 호출하면 루프가 종료됩니다.
    """
    period           = 1.0 / float(fps)
    stop_event       = threading.Event()
    expected_serials = set(CAM_SERIALS.values()) if CAM_NAMES else set()
    serial_to_name   = {v: k for k, v in CAM_SERIALS.items()}

    # ── stale frame / 카메라 누락 허용 한계 ──────────────────
    # GC 일시정지나 USB 지터로 동기화가 잠시 지연될 수 있음.
    # 이 시간(초)을 초과하면 stale/missing이어도 현재 프레임을 수용.
    _STALE_TIMEOUT_SEC   = 0.5   # 같은 timestamp가 이 시간 이상 반복되면 수용
    _MISSING_TIMEOUT_SEC = 2.0   # 카메라 누락이 이 시간 이상 지속되면 있는 것만 사용

    def _loop():
        logger.info("[DataLogger] 루프 시작")
        next_t              = time.perf_counter()
        record_start_t      = time.perf_counter()
        last_primary_ts     = None
        warmup_unique       = 0
        last_warmup_primary = None
        last_cam_warn_t     = 0.0
        last_wu_warn_t      = 0.0
        stale_since_t       = None  # stale frame이 처음 감지된 시각
        missing_since_t     = None  # 카메라 누락이 처음 감지된 시각

        while not stop_event.is_set():
            # ── 카메라 프레임 수집 ────────────────────────────
            frames = _realsense.get_frames() if _realsense else {}
            if not frames:
                frames = {}

            frame       = None
            depth       = None
            frame_stamp = None

            if PRIMARY_SERIAL and PRIMARY_SERIAL in frames:
                try:
                    primary_data = frames[PRIMARY_SERIAL]
                    frame        = primary_data.color
                    depth        = primary_data.depth
                    frame_stamp  = primary_data.t
                except Exception as e:
                    logger.warning(f"[DataLogger] 주 카메라 프레임 오류: {e}")

            # ── 카메라 동기화 검사 ────────────────────────────
            if expected_serials:
                missing = expected_serials - set(frames.keys())
                if missing:
                    now_t = time.perf_counter()
                    if missing_since_t is None:
                        missing_since_t = now_t

                    missing_elapsed = now_t - missing_since_t
                    if missing_elapsed < _MISSING_TIMEOUT_SEC:
                        # 아직 타임아웃 전 — skip
                        if now_t - last_cam_warn_t > 1.0:
                            names = [serial_to_name.get(s, s) for s in sorted(missing)]
                            logger.warning(f"[DataLogger] 카메라 누락: {names} ({missing_elapsed:.1f}s)")
                            last_cam_warn_t = now_t
                        next_t += period
                        time.sleep(max(0.0, next_t - time.perf_counter()))
                        continue
                    else:
                        # 타임아웃 초과 — 있는 카메라만으로 진행
                        if now_t - last_cam_warn_t > 5.0:
                            names = [serial_to_name.get(s, s) for s in sorted(missing)]
                            logger.warning(
                                f"[DataLogger] 카메라 누락 타임아웃 ({missing_elapsed:.1f}s) — "
                                f"있는 카메라만 사용: {[serial_to_name.get(s, s) for s in frames.keys()]}"
                            )
                            last_cam_warn_t = now_t
                else:
                    missing_since_t = None  # 모든 카메라 정상 — 타이머 리셋

                if frame_stamp is None:
                    next_t += period
                    time.sleep(max(0.0, next_t - time.perf_counter()))
                    continue

                # stale frame 검사 (GC 지연 시 영구 skip 방지)
                if last_primary_ts is not None and frame_stamp <= last_primary_ts + 1e-6:
                    now_t = time.perf_counter()
                    if stale_since_t is None:
                        stale_since_t = now_t

                    stale_elapsed = now_t - stale_since_t
                    if stale_elapsed < _STALE_TIMEOUT_SEC:
                        next_t += period
                        time.sleep(max(0.0, next_t - time.perf_counter()))
                        continue
                    else:
                        # 타임아웃 초과 — stale이어도 현재 프레임 수용
                        logger.warning(
                            f"[DataLogger] stale frame 타임아웃 ({stale_elapsed:.1f}s) — "
                            f"현재 프레임 수용 (ts={frame_stamp:.3f})"
                        )
                        stale_since_t = None
                else:
                    stale_since_t = None  # fresh frame — 타이머 리셋

                # 카메라 워밍업
                if not state.camera_warmup_done:
                    if frame_stamp is not None and (
                        last_warmup_primary is None or frame_stamp > last_warmup_primary + 1e-6
                    ):
                        warmup_unique       += 1
                        last_warmup_primary  = frame_stamp

                    elapsed     = time.perf_counter() - record_start_t
                    warmup_done = (
                        elapsed >= CAMERA_WARMUP_SECS
                        and warmup_unique >= CAMERA_WARMUP_FRAMES
                    )
                    if not warmup_done:
                        now_t = time.perf_counter()
                        if now_t - last_wu_warn_t > 1.0:
                            logger.info(
                                f"[DataLogger] 카메라 워밍업 "
                                f"({elapsed:.1f}s/{CAMERA_WARMUP_SECS:.1f}s, "
                                f"{warmup_unique}/{CAMERA_WARMUP_FRAMES}프레임)"
                            )
                            last_wu_warn_t = now_t
                        next_t += period
                        time.sleep(max(0.0, next_t - time.perf_counter()))
                        continue
                    state.camera_warmup_done = True
                    logger.info("[DataLogger] 카메라 워밍업 완료")

            # ── 로봇 관절 위치 읽기 ───────────────────────────
            with state.lock:
                robot_pos = state.robot_joint_positions.copy() if state.robot_joint_positions.size > 0 else None
                master_q  = state.master_arm_q.copy() if state.master_arm_q is not None else None
                grip_tgt  = state.gripper_target.copy()   # 0=CLOSED, 1=OPEN
                grip      = state.gripper_state.copy()    # 0=CLOSED, 1=OPEN (state_poll_thread 캐시)
                # ※ 직접 _gripper.get_state() UDP 호출 제거:
                #   state_poll_thread_fn이 ~30Hz로 폴링 후 SharedState에 캐시.
                #   이로써 set_normalized_target ACK와 소켓 경쟁/stale packet 문제 해결.

            if robot_pos is None:
                next_t += period
                time.sleep(max(0.0, next_t - time.perf_counter()))
                continue

            # ── robot_target_joints 구성 ──────────────────────
            robot_target_joints = robot_pos.copy()
            if master_q is not None and robot_model is not None:
                right_idx = list(robot_model.right_arm_idx)
                left_idx  = list(robot_model.left_arm_idx)
                if len(right_idx) == 7 and len(master_q) >= 14:
                    robot_target_joints[right_idx] = master_q[MASTER_RIGHT_SLICE]
                if len(left_idx) == 7 and len(master_q) >= 14:
                    robot_target_joints[left_idx] = master_q[MASTER_LEFT_SLICE]

            # ── PCD 생성 ──────────────────────────────────────
            pcd_points = None
            pcd_colors = None
            if frame is not None and depth is not None:
                try:
                    H_d, W_d = depth.shape
                    intrinsics = (
                        REALSENSE_D435_INTRINSICS_848x480
                        if (W_d == 848 and H_d == 480)
                        else REALSENSE_D435_INTRINSICS
                    )
                    depth_clipped = np.where(depth > 3000, 0, depth)
                    pcd_points, pcd_colors = rgbd_to_pointcloud(
                        frame, depth_clipped,
                        intrinsics["fx"], intrinsics["fy"],
                        intrinsics["cx"], intrinsics["cy"],
                    )
                except Exception as e:
                    logger.warning(f"[DataLogger] PCD 생성 실패: {e}")

            # ── H5 저장 ───────────────────────────────────────
            sample_ts = float(frame_stamp) if frame_stamp is not None else time.time()
            h5_writer.update_previous_target(robot_pos)

            data = {
                "ts":                  sample_ts,
                "robot_position":      robot_pos,
                "robot_target_joints": robot_target_joints,
                "gripper_state":       grip,      # 0=CLOSED, 1=OPEN
                "gripper_target":      grip_tgt,  # 0=CLOSED, 1=OPEN (동일 규칙)
                "base_state":          np.zeros(3, dtype=float),
                "pcd_points":          pcd_points,
                "pcd_colors":          pcd_colors,
            }

            for cam_name in CAM_NAMES:
                serial = CAM_SERIALS[cam_name]
                if serial not in frames:
                    continue
                cam_data = frames[serial]
                data[f"{cam_name}_rgb"]       = cam_data.color
                data[f"{cam_name}_rgb_ts"]    = cam_data.t
                if cam_data.depth is not None:
                    data[f"{cam_name}_depth"]    = cam_data.depth
                    data[f"{cam_name}_depth_ts"] = cam_data.t

            h5_writer.put(data)

            if frame_stamp is not None:
                last_primary_ts = frame_stamp

            # ── FPS 페이싱 ────────────────────────────────────
            next_t += period
            slack = next_t - time.perf_counter()
            if slack > 0:
                time.sleep(slack)
            else:
                next_t = time.perf_counter()

        logger.info("[DataLogger] 루프 종료")

    t = threading.Thread(target=_loop, name="data_logger", daemon=True)
    t.start()
    logger.info(f"[DataLogger] 시작 ({fps} FPS)")
    return stop_event


print("✅ start_data_logger 함수 정의 완료")
print("   gripper_state / gripper_target 규칙: 0=CLOSED, 1=OPEN (통일)")

In [ ]:
# [Step 7-2] 데이터 수집 UI 위젯 정의 (녹화/중지/폐기/초기복귀 버튼)
# ══════════════════════════════════════════════════════════
# 📹 카메라 라이브 프리뷰 (카메라가 있을 때만 동작)
# ══════════════════════════════════════════════════════════

_preview_stop = threading.Event()

def start_camera_preview(
    _realsense: Optional[MultiRealsense],
    primary_serial: Optional[str],
    preview_widget: widgets.Image,
    stop_evt: threading.Event,
    preview_fps: int = 10,
):
    """백그라운드에서 카메라 프레임을 preview_widget에 업데이트."""
    if _realsense is None or primary_serial is None:
        return

    def _run():
        period = 1.0 / preview_fps
        while not stop_evt.is_set():
            try:
                frames = _realsense.get_frames()
                if frames and primary_serial in frames:
                    img = frames[primary_serial].color
                    if img is not None:
                        _, buf = cv2.imencode(
                            ".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 70]
                        )
                        preview_widget.value = buf.tobytes()
            except Exception:
                pass
            time.sleep(period)

    threading.Thread(target=_run, name="preview", daemon=True).start()


# ══════════════════════════════════════════════════════════
# 🎛️  데이터 수집 UI (ipywidgets)
# ══════════════════════════════════════════════════════════

# ── 위젯 정의 ─────────────────────────────────────────────
_status_html   = widgets.HTML(value='<b style="font-size:1.1em">⚫ 대기 중</b>')
_episode_label = widgets.Label(
    value=f"저장된 에피소드: {len([d for d in os.listdir(DEMO_ROOT) if d.startswith('episode_')])}개"
)
_path_label    = widgets.Label(value="저장 경로: —")

_btn_start   = widgets.Button(
    description="▶ Start Recording",
    button_style="success",
    layout=widgets.Layout(width="180px", height="36px"),
)
_btn_stop    = widgets.Button(
    description="⏹ Stop & Save",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(width="160px", height="36px"),
)
_btn_reset   = widgets.Button(
    description="🔄 초기 자세 복귀",
    button_style="info",
    layout=widgets.Layout(width="160px", height="36px"),
    tooltip="로봇 + Master Arm을 초기 자세로 이동 후 다음 에피소드 준비",
)

_preview_img = widgets.Image(
    format="jpeg",
    layout=widgets.Layout(width="320px", height="240px"),
)

_joint_out = widgets.Output()


def _refresh_joint_display():
    """관절 값 표시 주기적 업데이트."""
    while True:
        if not shared_state.is_teleop_active:
            time.sleep(1.0)
            continue
        with shared_state.lock:
            q    = shared_state.master_arm_q
            rpos = shared_state.robot_joint_positions
            gtgt = shared_state.gripper_target.copy()
            r_active = shared_state.unlock_right_active
            l_active = shared_state.unlock_left_active
        _joint_out.clear_output(wait=True)
        with _joint_out:
            r_icon = "🔓" if r_active else "🔒"
            l_icon = "🔓" if l_active else "🔒"
            if q is not None:
                print(f"Master Arm  오른팔 {r_icon}: {np.round(np.rad2deg(q[MASTER_RIGHT_SLICE]), 1)}°")
                print(f"            왼팔   {l_icon}: {np.round(np.rad2deg(q[MASTER_LEFT_SLICE]),  1)}°")
            if rpos.size > 0 and robot_model is not None:
                ridx = list(robot_model.right_arm_idx)
                lidx = list(robot_model.left_arm_idx)
                print(f"Robot state 오른팔: {np.round(np.rad2deg(rpos[ridx]), 1)}°")
                print(f"            왼팔:   {np.round(np.rad2deg(rpos[lidx]),  1)}°")
            print(f"그리퍼 타겟: right={gtgt[0]:.2f}, left={gtgt[1]:.2f}")
        time.sleep(0.5)


threading.Thread(target=_refresh_joint_display, daemon=True, name="joint_disp").start()


# ── 버튼 콜백 ─────────────────────────────────────────────

def _set_all_buttons_enabled(enabled: bool):
    """모든 제어 버튼을 일괄 활성화/비활성화."""
    _btn_start.disabled  = not enabled
    _btn_reset.disabled  = not enabled
    if enabled:
        _btn_stop.disabled    = True


def _on_start_recording(b):
    if shared_state.is_recording:
        return

    output_path = get_next_h5_path(DEMO_ROOT)
    hw = H5Writer(path=output_path, flush_every=60, flush_secs=1.0).start()
    stop_evt = start_data_logger(hw, shared_state, gripper, realsense, fps=REC_FPS)

    shared_state.h5_writer            = hw
    shared_state.rec_stop_event        = stop_evt
    shared_state.is_recording          = True
    shared_state.current_episode_path  = output_path
    shared_state.camera_warmup_done    = False

    _btn_start.disabled   = True
    _btn_reset.disabled   = True
    _btn_stop.disabled    = False
    ep_name = os.path.basename(os.path.dirname(output_path))
    _status_html.value = f'<b style="color:red;font-size:1.1em">🔴 녹화 중 — {ep_name}</b>'
    _path_label.value  = f"저장 경로: {output_path}"
    logger.info(f"[UI] 녹화 시작: {output_path}")


def _stop_recording_internal() -> Optional[str]:
    """녹화 중지 및 저장. 저장된 경로 반환."""
    if not shared_state.is_recording:
        return None
    if shared_state.rec_stop_event:
        shared_state.rec_stop_event.set()
    if shared_state.h5_writer:
        shared_state.h5_writer.stop()
    saved_path = shared_state.current_episode_path
    shared_state.is_recording = False
    _btn_stop.disabled    = True
    return saved_path


def _on_stop_recording(b):
    saved_path = _stop_recording_internal()
    if saved_path is None:
        return
    _btn_start.disabled = False
    _btn_reset.disabled = False
    _status_html.value = '<b style="color:green;font-size:1.1em">✅ 저장 완료 — 초기 자세로 복귀하거나 다음 에피소드를 시작하세요.</b>'
    _path_label.value  = f"저장 경로: {saved_path}"
    ep_count = len([d for d in os.listdir(DEMO_ROOT) if d.startswith("episode_")])
    _episode_label.value = f"저장된 에피소드: {ep_count}개"
    logger.info(f"[UI] 녹화 저장: {saved_path}")



def _on_reset_pose(b):
    """
    로봇 + Master Arm 초기 자세 복귀 후 텔레오퍼레이션 재시작.

    동작 순서:
        1. 녹화 중이면 자동 저장
        2. 모든 버튼 비활성화
        3. 텔레오퍼레이션 중단 (명령 스트림 해제)
        4. 로봇 초기 자세 이동 (send_command — 스트림 없이 blocking)
        5. Master Arm homing
        6. 텔레오퍼레이션 재시작 (새 명령 스트림 생성)
        7. 버튼 다시 활성화

    ※ 텔레오퍼레이션 스트림(priority=1)이 활성화된 상태에서는
       robot.send_command()가 무시되므로, 반드시 스트림을 먼저 해제해야 합니다.
    """
    def _do_reset():
        try:
            # 1) 녹화 중이면 자동 저장
            if shared_state.is_recording:
                saved = _stop_recording_internal()
                if saved:
                    logger.info(f"[Reset] 자동 저장: {saved}")
                    ep_count = len([d for d in os.listdir(DEMO_ROOT) if d.startswith("episode_")])
                    _episode_label.value = f"저장된 에피소드: {ep_count}개"

            # 2) 버튼 비활성화
            _set_all_buttons_enabled(False)
            _btn_stop.disabled    = True
            # 3) 텔레오퍼레이션 중단 → 명령 스트림 해제
            _status_html.value = '<b style="color:blue">⏳ 텔레오퍼레이션 스트림 해제 중...</b>'
            stop_teleoperation()
            time.sleep(0.3)  # 스트림 해제 안정화 대기

            # 4) 로봇 초기 자세 이동 (blocking send_command)
            _status_html.value = '<b style="color:blue">⏳ 로봇 초기 자세 이동 중... (약 5초)</b>'
            ok_robot = move_to_init_pose(minimum_time=5.0)
            if not ok_robot:
                logger.warning("[Reset] 로봇 초기 자세 이동 완전 수렴 실패 (계속 진행)")

            # 5) Master Arm homing
            _status_html.value = '<b style="color:blue">⏳ Master Arm homing 중...</b>'
            ok_ma = move_master_arm_to_init_pose(
                timeout=20.0,
                threshold_deg=5.0,
                max_speed_deg_per_sec=30.0,
            )
            if not ok_ma:
                logger.warning("[Reset] Master Arm homing 완전 수렴 실패 (계속 진행)")

            # 6) 텔레오퍼레이션 재시작 (새 스트림 + hold 위치 자동 설정)
            _status_html.value = '<b style="color:blue">⏳ 텔레오퍼레이션 재시작 중...</b>'
            start_teleoperation()
            time.sleep(0.5)

            _status_html.value = (
                '<b style="color:green;font-size:1.1em">'
                '✅ 초기 자세 복귀 완료 — ▶ Start Recording으로 다음 에피소드를 시작하세요.'
                '</b>'
            )
            _path_label.value = "저장 경로: —"
            logger.info("[Reset] 초기 자세 복귀 + 텔레오퍼레이션 재시작 완료")

        except Exception as e:
            _status_html.value = f'<b style="color:red">❌ 복귀 중 오류: {e}</b>'
            logger.error(f"[Reset] 오류: {e}", exc_info=True)
        finally:
            # 7) 버튼 다시 활성화
            _btn_start.disabled   = False
            _btn_reset.disabled   = False

    threading.Thread(target=_do_reset, daemon=True, name="reset_pose").start()


_btn_start.on_click(_on_start_recording)
_btn_stop.on_click(_on_stop_recording)
_btn_reset.on_click(_on_reset_pose)

print("✅ UI 위젯 정의 완료 — 다음 셀에서 display()로 표시합니다.")

In [ ]:
# [Step 7-3] UI 표시 및 카메라 라이브 프리뷰 시작
# ══════════════════════════════════════════════════════════
# 📺  UI 표시 + 카메라 프리뷰 시작
# ══════════════════════════════════════════════════════════

# 카메라 프리뷰 시작 (카메라가 있을 때만)
_preview_stop.clear()
start_camera_preview(realsense, PRIMARY_SERIAL, _preview_img, _preview_stop, preview_fps=8)

# ── 레이아웃 구성 ─────────────────────────────────────────
_header  = widgets.HTML(
    "<h3 style='margin:4px 0'>🦾 Master Arm 텔레오퍼레이션 — 데이터 수집</h3>"
)
_divider = widgets.HTML("<hr style='margin:4px 0'>")

# 버튼 행 1: 녹화 제어
_rec_row = widgets.HBox(
    [_btn_start, _btn_stop],
    layout=widgets.Layout(margin="4px 0"),
)
# 버튼 행 2: 초기 자세 복귀
_reset_row = widgets.HBox(
    [_btn_reset],
    layout=widgets.Layout(margin="4px 0"),
)

_workflow_html = widgets.HTML(
    "<small style='color:#555'>"
    "▶ Start → Demo → ⏹ Stop & Save → 🔄 Reset Pose → Repeat"
    "</small>"
)

_info_box = widgets.VBox(
    [_status_html, _path_label, _episode_label],
    layout=widgets.Layout(margin="6px 0"),
)

_left_panel = widgets.VBox(
    [_header, _divider, _rec_row, _reset_row, _workflow_html, _info_box, _joint_out],
    layout=widgets.Layout(min_width="460px", padding="6px"),
)

_right_panel = widgets.VBox(
    [
        widgets.HTML("<b>📷 메인 카메라 프리뷰</b>"),
        _preview_img if realsense else widgets.HTML("<i>카메라 없음</i>"),
    ],
    layout=widgets.Layout(padding="6px"),
)

_ui = widgets.HBox([_left_panel, _right_panel])
display(_ui)


---
## 에피소드 현황 확인

수집된 에피소드 목록과 파일 크기를 확인합니다.


In [ ]:
# [확인] 저장된 에피소드 목록 및 파일 크기 확인
import h5py

def show_episodes(demo_root: str):
    episodes = sorted(
        [d for d in os.listdir(demo_root) if d.startswith("episode_")],
        key=lambda x: int(x.split("_")[1]) if x.split("_")[1].isdigit() else 0,
    )
    if not episodes:
        print(f"수집된 에피소드 없음 ({demo_root})")
        return

    print(f"{'에피소드':<15} {'파일':<35} {'크기(MB)':>10} {'스텝 수':>8}")
    print("─" * 75)
    total_steps = 0
    for ep in episodes:
        ep_dir = os.path.join(demo_root, ep)
        h5_files = [f for f in os.listdir(ep_dir) if f.endswith(".h5")]
        for hf in h5_files:
            fp = os.path.join(ep_dir, hf)
            sz = os.path.getsize(fp) / 1e6
            steps = "?"
            try:
                with h5py.File(fp, "r") as f:
                    if "samples" in f and "time" in f["samples"]:
                        steps = len(f["samples"]["time"])
                        total_steps += steps
            except Exception:
                pass
            print(f"{ep:<15} {hf:<35} {sz:>10.2f} {str(steps):>8}")

    print("─" * 75)
    print(f"총 에피소드: {len(episodes)}개  |  총 스텝: {total_steps}")


show_episodes(DEMO_ROOT)


---
## 🧹 Cleanup — 시스템 종료

**모든 작업이 끝나면** 아래 셀을 실행하여 시스템을 안전하게 종료합니다.

| 단계 | 동작 |
|---|---|
| 1 | 진행 중인 녹화 자동 저장 |
| 2 | 텔레오퍼레이션 스레드 중단 |
| 3 | Master Arm 서버에 stop 명령 + 소켓 닫기 |
| 4 | 카메라 정지 |
| 5 | 로봇 Control Manager 비활성화 (파워 OFF 아님) |

> ⚠️ 로봇 전원은 꺼지지 않습니다. Control Manager만 해제되어 관절이 자유 상태가 됩니다.  
> 로봇 전원을 끄려면 별도로 `robot.power_off(".*")`를 실행하세요.

In [ ]:
# [Cleanup] 전체 시스템 종료 (녹화 저장 → Teleop 중단 → MA → 카메라 → CM 비활성화)
def full_cleanup():
    """
    모든 하드웨어 및 스레드를 안전하게 종료합니다.

    - 로봇 전원은 끄지 않고 Control Manager만 비활성화합니다.
    - Master Arm 서버에 stop 명령을 보내고 소켓을 닫습니다.
    """
    global master_arm
    logger.info("=== 시스템 종료 시작 ===")

    # 1) 진행 중인 녹화 저장
    if shared_state.is_recording:
        logger.info("녹화 중 → 자동 저장")
        if shared_state.rec_stop_event:
            shared_state.rec_stop_event.set()
        if shared_state.h5_writer:
            shared_state.h5_writer.stop()
        shared_state.is_recording = False

    # 2) 텔레오퍼레이션 정지
    stop_teleoperation()

    # 3) Master Arm 서버에 stop 명령 + 소켓 닫기
    if master_arm is not None:
        try:
            master_arm.stop()
            logger.info("Master Arm 서버에 stop 명령 전송")
            time.sleep(0.3)  # 서버 idle 전환 대기
        except Exception as e:
            logger.warning(f"Master Arm stop 오류: {e}")
        try:
            master_arm.close()
            logger.info("Master Arm 소켓 닫기 완료")
        except Exception as e:
            logger.warning(f"Master Arm 소켓 닫기 실패: {e}")
        master_arm = None

    # 4) 카메라 정지
    stop_cameras()

    # 5) 로봇 Control Manager 비활성화 (전원 OFF 아님)
    if robot is not None:
        try:
            robot.disable_control_manager()
            logger.info("로봇 Control Manager 비활성화 완료")
        except Exception as e:
            logger.warning(f"Control Manager 비활성화 실패: {e}")

    logger.info("=== 시스템 종료 완료 ===")
    print("✅ 전체 시스템 종료 완료")
    print("   - 로봇 Control Manager 비활성화됨 (전원은 유지)")
    print("   - Master Arm 서버 idle 모드 전환 + 소켓 닫힘")
    print("   ℹ️  Robot PC의 master_arm_server.py는 계속 실행 중입니다.")
    print("      필요 시 Ctrl+C로 서버를 종료하세요.")


full_cleanup()